# Unified Experiment Notebook: Subsampling & Margin Control

## Linear Perception Task — Single-Shortcut (E1-E6) & Two-Shortcut (E7)

**Paper:** *Subsampling and Margin Control: Overparameterisation as the Central Failure Mode in Shortcut Mitigation

***Purpose:** Run all experiments, save results (CSV) and model checkpoints to Google Drive.

No plots — analysis notebook is separate.

### Experiment Index
| # | Name | Runs | Key Question |
|---|------|------|-------------|
| E1 | Baseline | ~12 | Reproduce Puli et al. Fig 1 + GB-Down |
| E2 | Random subsampling grid | 240 | WGA vs ratio, rho, method |
| E3 | Group-balanced analysis | ~24 | GB-Down at all rho values |
| E4 | Dataset size ablation | 216 | Confirm d/n=1 phase boundary |
| E5 | Overfitting analysis | 140 | Generalisation gap vs ratio |
| E6 | B-scaling | ~60 | B-invariance of MARG-CTRL |
| E7a | Two-shortcut baseline | 15 | Confirm Theorems 1', 2' |
| E7b | Per-shortcut balancing | 30 | Partial elimination (Prop 4.1) |
| E7c | Four-group balancing | 15 | Overparameterisation under Bal-4G |
| E7d | Two-shortcut size ablation | 72 | Per-shortcut phase transitions |
| E7e | Full two-shortcut grid | 60 | Strategy x method comparison |



In [1]:
# ============================================================
# Cell 1: Setup — Install deps, mount Drive, create directories
# ============================================================
import subprocess, sys

# Install any missing packages (Colab has most already)
for pkg in ['seaborn', 'pandas']:
    try:
        __import__(pkg)
    except ImportError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])

# Mount Google Drive
try:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_ROOT = '/content/drive/MyDrive/shortcut_experiments'
    IN_COLAB = True
except ImportError:
    DRIVE_ROOT = './shortcut_experiments'
    IN_COLAB = False

import os
os.makedirs(f'{DRIVE_ROOT}/csv', exist_ok=True)
os.makedirs(f'{DRIVE_ROOT}/checkpoints', exist_ok=True)
os.makedirs(f'{DRIVE_ROOT}/logs', exist_ok=True)
print(f'Drive root: {DRIVE_ROOT}')
print(f'Running in Colab: {IN_COLAB}')

Mounted at /content/drive
Drive root: /content/drive/MyDrive/shortcut_experiments
Running in Colab: True


## Section 0: Global Configuration
All hyperparameters from paper Section 5.2. Change `RUN_SINGLE_SHORTCUT` / `RUN_TWO_SHORTCUT` flags to selectively run experiment groups.

In [2]:
# ============================================================
# Cell 2: Imports and global parameters
# ============================================================
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import pandas as pd
import time, json, os, warnings
from collections import OrderedDict
warnings.filterwarnings('ignore')
from collections import Counter
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import os, glob
from torchvision import transforms, models
import copy
from sklearn.preprocessing import StandardScaler


# Reproducibility
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# ── Experiment flags ──
RUN_SINGLE_SHORTCUT = True   # E1–E6
RUN_TWO_SHORTCUT    = True   # E7a–E7e

# ── Standard DGP parameters (paper Section 5.2) ──
D       = 300       # feature dimension
B       = 10        # shortcut magnitude (single-shortcut)
B1, B2  = 10, 10    # shortcut magnitudes (two-shortcut)
RHO     = 0.9       # default single-shortcut correlation
N_TRAIN = 1000      # default training set size
N_TEST  = 2000      # test set size
RHO1_TRAIN, RHO2_TRAIN = 0.9, 0.7  # two-shortcut correlations
RHO1_TEST,  RHO2_TEST  = 0.1, 0.3  # flipped for test

# ── Training ──
EPOCHS  = 10000     # convergence epochs (linear model)
LR      = 0.01      # learning rate (Adam)

# ── Derived constants ──
N_CRIT_SINGLE = D / (2 * (1 - RHO))                     # 1500
N_CRIT_Z1     = D / (2 * (1 - RHO1_TRAIN))              # 1500
N_CRIT_Z2     = D / (2 * (1 - RHO2_TRAIN))              # 500

print(f'Device: {device}')
print(f'DGP: d={D}, B={B}, rho={RHO}, n_train={N_TRAIN}')
print(f'Two-shortcut: B1={B1}, B2={B2}, rho1={RHO1_TRAIN}, rho2={RHO2_TRAIN}')
print(f'n_critical (single, GB-Down): {N_CRIT_SINGLE:.0f}')
print(f'n_critical (z1): {N_CRIT_Z1:.0f}, n_critical (z2): {N_CRIT_Z2:.0f}')

Device: cuda
DGP: d=300, B=10, rho=0.9, n_train=1000
Two-shortcut: B1=10, B2=10, rho1=0.9, rho2=0.7
n_critical (single, GB-Down): 1500
n_critical (z1): 1500, n_critical (z2): 500


## Section 1: Infrastructure
### 1.1 Data Generating Processes

In [3]:
# ============================================================
# Cell 3: Data generating processes (single + two shortcut)
# ============================================================

def generate_linear_data(n, d, rho, B, seed=None):
    """Single-shortcut linear perception task (Puli et al. 2023, Def 3.1).

    x = [B*z, y, delta] where delta ~ N(0, I_{d-2}).
    Groups: 0 = shortcut (z==y), 1 = leftover (z!=y).
    """
    rng = np.random.RandomState(seed)
    y = rng.choice([-1.0, 1.0], size=n)
    flip = rng.binomial(1, 1 - rho, size=n)
    z = y * (1 - 2 * flip)
    delta = rng.randn(n, d - 2) # n x d-2
    X = np.column_stack([B * z, y, delta]).astype(np.float32)
    groups = (y != z).astype(int)  # 0=shortcut, 1=leftover
    return X, y.astype(np.float32), z.astype(np.float32), groups # x contains [B*z, y, delta], groups contains wither 0 where shortcut agrees with label or 1 where shortcut disagrees with label


def generate_two_shortcut_data(n, d, rho1, rho2, B1, B2, seed=None):
    """Two-shortcut linear perception task (paper Def 4.1).

    x = [B1*z1, B2*z2, y, delta] where delta ~ N(0, I_{d-3}).
    Groups: 0=(++), 1=(+-), 2=(-+), 3=(--) by (z1,z2) agreement with y.
    """
    rng = np.random.RandomState(seed)
    y = rng.choice([-1.0, 1.0], size=n)
    flip1 = rng.binomial(1, 1 - rho1, size=n)
    flip2 = rng.binomial(1, 1 - rho2, size=n)
    z1 = y * (1 - 2 * flip1) # shortcut 1
    z2 = y * (1 - 2 * flip2) # shortcut 2
    delta = rng.randn(n, d - 3)
    X = np.column_stack([B1 * z1, B2 * z2, y, delta]).astype(np.float32)
    agree1 = (y == z1)
    agree2 = (y == z2)
    groups = np.zeros(n, dtype=int)
    groups[agree1 & agree2]   = 0  # ++
    groups[agree1 & ~agree2]  = 1  # +-
    groups[~agree1 & agree2]  = 2  # -+
    groups[~agree1 & ~agree2] = 3  # --
    return X, y.astype(np.float32), z1.astype(np.float32), z2.astype(np.float32), groups # 4 groups when 2 shortcuts

print('DGP functions ready.')

DGP functions ready.


### 1.2 Loss Functions (paper Section 3.5, Table in Section 5.3)

In [4]:
# ============================================================
# Cell 4: Loss functions — ERM, sigma-Damp, SD, Marg-Log
# ============================================================

class LogLoss(nn.Module):
    """Standard binary log-loss (ERM)."""
    def forward(self, logits, targets): # target = my label, logit = my predicytion
        margins = targets * logits.squeeze()
        return torch.log(1 + torch.exp(-margins)).mean() # L=N1​i=1∑N​log(1+e−yi​⋅f(xi​))


class SigmaDampedLoss(nn.Module):
    """Sigmoid-damped loss (sigma-Damp). Temperature T controls margin cap."""
    def __init__(self, temperature=2.0):
        super().__init__()
        self.T = temperature # param controls
    def forward(self, logits, targets):
        f = logits.squeeze()
        margins = targets * f
        damped = targets * (1 - torch.sigmoid(margins / self.T)) * f
        return torch.log(1 + torch.exp(-damped)).mean()


class SpectralDecoupling(nn.Module):
    """Spectral Decoupling: log-loss + lambda * ||f||^2."""
    def __init__(self, lam=0.1):
        super().__init__()
        self.lam = lam # param
    def forward(self, logits, targets):
        f = logits.squeeze()
        return torch.log(1 + torch.exp(-targets * f)).mean() + self.lam * (f ** 2).mean()


class MargLogLoss(nn.Module):
    """Margin-Log: log-loss + lambda * log(1 + f^2)."""
    def __init__(self, lam=0.5):
        super().__init__()
        self.lam = lam
    def forward(self, logits, targets):
        f = logits.squeeze()
        return torch.log(1 + torch.exp(-targets * f)).mean() + self.lam * torch.log(1 + f ** 2).mean()


# Registries for easy iteration
LOSS_FNS = OrderedDict([
    ('ERM',      lambda: LogLoss()),
    ('sigma-Damp', lambda: SigmaDampedLoss(temperature=2.0)),
    ('SD',       lambda: SpectralDecoupling(lam=0.1)),
    ('Marg-Log', lambda: MargLogLoss(lam=0.5)),
])

# Subset used in most experiments (sigma-Damp only in E2)
LOSS_FNS_CORE = OrderedDict([
    ('ERM',      LOSS_FNS['ERM']),
    ('SD',       LOSS_FNS['SD']),
    ('Marg-Log', LOSS_FNS['Marg-Log']),
])

print('Loss functions:', list(LOSS_FNS.keys()))

Loss functions: ['ERM', 'sigma-Damp', 'SD', 'Marg-Log']


### 1.3 Model and Training

In [5]:
# ============================================================
# Cell 5: Linear model + training loop
# ============================================================

class LinearModel(nn.Module):
    """Linear classifier f(x) = w^T x, no bias (paper Section 5.3)."""
    def __init__(self, d):
        super().__init__()
        self.linear = nn.Linear(d, 1, bias=False)
        nn.init.normal_(self.linear.weight, std=0.01) # intialise weights
    def forward(self, x):
        return self.linear(x)
    def get_weights(self):
        return self.linear.weight.data.cpu().numpy().flatten()

def train_model(model, X_train, y_train, loss_fn, epochs=EPOCHS, lr=LR):
    """Full-batch Adam training to convergence."""
    model = model.to(device)
    X_t = torch.tensor(X_train, dtype=torch.float32).to(device)
    y_t = torch.tensor(y_train, dtype=torch.float32).to(device)
    optimizer = optim.Adam(model.parameters(), lr=lr) # adam opt
    for _ in range(epochs):
        optimizer.zero_grad()
        loss = loss_fn(model(X_t), y_t)
        loss.backward()
        optimizer.step()
    return model

print('Model + training ready.')

Model + training ready.


### 1.4 Evaluation & Weight Analysis

In [6]:
# ============================================================
# Cell 6: Evaluation functions (single + two shortcut)
# ============================================================

def evaluate_single(model, X, y, groups, B_val):
    """Evaluate single-shortcut model. Returns dict of all key metrics."""
    model.eval()
    with torch.no_grad():
        X_t = torch.tensor(X, dtype=torch.float32).to(device)
        logits = model(X_t).cpu().numpy().flatten()
    preds = np.sign(logits) # get sign of logits for predictions
    preds[preds == 0] = 1 # where preds is 0, change to 1
    margins = y * logits # margin = y * f(x)

    # Per-group accuracy
    idx_s = groups == 0  # shortcut group
    idx_l = groups == 1  # leftover group
    acc = float(np.mean(preds == y))
    acc_s = float(np.mean(preds[idx_s] == y[idx_s])) if idx_s.sum() > 0 else np.nan # acc of shortcut group
    acc_l = float(np.mean(preds[idx_l] == y[idx_l])) if idx_l.sum() > 0 else np.nan # acc of leftover group
    wga = min(v for v in [acc_s, acc_l] if not np.isnan(v)) # get minimum accuracies for any group

    # Weight analysis
    w = model.get_weights()
    w_z, w_y = w[0], w[1]
    B_wz = B_val * w_z # weight of shortcut
    sr = abs(B_wz) / (abs(w_y) + 1e-10) # weight of shortcut * B / weight of true feature
    w_noise_norm = float(np.linalg.norm(w[2:]))

    # Margin stats
    margin_gap = abs(float(margins[idx_s].mean()) - float(margins[idx_l].mean())) if idx_s.sum() > 0 and idx_l.sum() > 0 else np.nan # mean of margin of shortcut - mean of leftover
    train_margin_std = float(np.std(margins))

    return {
        'acc': acc, 'acc_shortcut': acc_s, 'acc_leftover': acc_l, 'wga': wga,
        'B_wz': float(B_wz), 'wy': float(w_y), 'shortcut_ratio': float(sr),
        'shortcut_dominates': int(sr > 1.0), 'w_noise_norm': w_noise_norm,
        'margin_gap': margin_gap, 'margin_std': train_margin_std,
    }


def evaluate_two_shortcut(model, X, y, z1, z2, groups, B1_val, B2_val):
    """Evaluate two-shortcut model with per-group accuracy."""
    model.eval()
    with torch.no_grad():
        X_t = torch.tensor(X, dtype=torch.float32).to(device)
        logits = model(X_t).cpu().numpy().flatten()
    preds = np.sign(logits)
    preds[preds == 0] = 1
    margins = y * logits

    acc = float(np.mean(preds == y))
    acc_g = {}
    margin_g = {}
    for g in range(4):
        idx = groups == g
        if idx.sum() > 0:
            acc_g[g] = float(np.mean(preds[idx] == y[idx]))
            margin_g[g] = float(margins[idx].mean())
        else:
            acc_g[g] = np.nan
            margin_g[g] = np.nan

    valid = [v for v in acc_g.values() if not np.isnan(v)]
    wga = min(valid) if valid else np.nan
    mv = [v for v in margin_g.values() if not np.isnan(v)]
    margin_gap = max(mv) - min(mv) if len(mv) > 1 else np.nan

    # Weight analysis
    w = model.get_weights()
    w_z1, w_z2, w_y = w[0], w[1], w[2]
    B1_wz1 = B1_val * w_z1
    B2_wz2 = B2_val * w_z2
    eps = 1e-10
    SR1 = abs(B1_wz1) / (abs(w_y) + eps)
    SR2 = abs(B2_wz2) / (abs(w_y) + eps)

    return {
        'acc': acc,
        'acc_g0': acc_g[0], 'acc_g1': acc_g[1], 'acc_g2': acc_g[2], 'acc_g3': acc_g[3],
        'wga': wga,
        'w_z1': float(w_z1), 'w_z2': float(w_z2), 'w_y': float(w_y),
        'B1_wz1': float(B1_wz1), 'B2_wz2': float(B2_wz2),
        'SR1': float(SR1), 'SR2': float(SR2),
        'shortcut1_dominates': int(SR1 > 1.0), 'shortcut2_dominates': int(SR2 > 1.0),
        'w_noise_norm': float(np.linalg.norm(w[3:])),
        'margin_gap': margin_gap,
        'margin_g0': margin_g[0], 'margin_g1': margin_g[1],
        'margin_g2': margin_g[2], 'margin_g3': margin_g[3],
    }

print('Evaluation functions ready.')

Evaluation functions ready.


### 1.5 Subsampling Strategies (paper Sections 3.6, 4.8, 5.4)

In [7]:
# Cell 7: Subsampling strategies (single + two shortcut)

# ── Single-shortcut strategies ──

def random_subsample(X, y, z, groups, ratio, seed=None):
    """Random subsample at rate `ratio`. Preserves k/n ratio (Lemma 3.2)."""
    rng = np.random.RandomState(seed)
    n_sub = max(int(len(y) * ratio), 2)
    idx = rng.choice(len(y), n_sub, replace=False) # choose randomly either from y or n_sub
    return X[idx], y[idx], z[idx], groups[idx]


def group_balanced_downsample(X, y, z, groups, seed=None):
    """GB-Down: equalise shortcut and leftover groups (Lemma 3.3).
    n_sub = 2k where k = |leftover|. Changes k/n -> 0.5."""
    rng = np.random.RandomState(seed)
    idx_s = np.where(groups == 0)[0]
    idx_l = np.where(groups == 1)[0]
    n_min = min(len(idx_s), len(idx_l)) # which group is smaller - shortcut or leftover?
    idx = np.concatenate([
        rng.choice(idx_s, n_min, replace=False), # sample n_min indices from shortcut group
        rng.choice(idx_l, n_min, replace=False),
    ])
    rng.shuffle(idx)
    return X[idx], y[idx], z[idx], groups[idx] # equal n_min samples of shortcut, leftover group


def random_subsample_to_size(X, y, z, groups, target_n, seed=None): # specify size target_n to subsample from y
    """Random subsample to exact size (matched-n control). Preserves k/n."""
    rng = np.random.RandomState(seed)
    target_n = min(target_n, len(y))
    idx = rng.choice(len(y), target_n, replace=False)
    return X[idx], y[idx], z[idx], groups[idx]


def group_balanced_oversample(X, y, z, groups, seed=None):
    """GB-Over: oversample minority to match majority. k/n=0.5, n_sub~2*n_majority."""
    rng = np.random.RandomState(seed)
    idx_s = np.where(groups == 0)[0]
    idx_l = np.where(groups == 1)[0]
    n_max = max(len(idx_s), len(idx_l)) # choose max group
    if len(idx_s) < n_max:
        idx_s = rng.choice(idx_s, n_max, replace=True)
    if len(idx_l) < n_max:
        idx_l = rng.choice(idx_l, n_max, replace=True)
    idx = np.concatenate([idx_s, idx_l]) # n_max samples of both shortcut, leftover groups
    rng.shuffle(idx)
    return X[idx], y[idx], z[idx], groups[idx]


# ── Two-shortcut strategies ──

def balance_shortcut1(X, y, z1, z2, groups, seed=None):
    """Bal-z1: equalise z1-agree vs z1-disagree (groups 0+1 vs 2+3)."""
    rng = np.random.RandomState(seed)
    idx_agree    = np.where(np.isin(groups, [0, 1]))[0] # where z1 agrees ( ++ and +-)
    idx_disagree = np.where(np.isin(groups, [2, 3]))[0] # where z1 disagrees (-- and -+)
    n_min = min(len(idx_agree), len(idx_disagree)) # which group is smaller?
    idx = np.concatenate([
        rng.choice(idx_agree, n_min, replace=False), # n_min samples of both
        rng.choice(idx_disagree, n_min, replace=False),
    ])
    rng.shuffle(idx)
    return X[idx], y[idx], z1[idx], z2[idx], groups[idx]


def balance_shortcut2(X, y, z1, z2, groups, seed=None):
    """Bal-z2: equalise z2-agree vs z2-disagree (groups 0+2 vs 1+3)."""
    rng = np.random.RandomState(seed)
    idx_agree    = np.where(np.isin(groups, [0, 2]))[0] # where z2 agrees
    idx_disagree = np.where(np.isin(groups, [1, 3]))[0] # where z2 disagrees
    n_min = min(len(idx_agree), len(idx_disagree))
    idx = np.concatenate([
        rng.choice(idx_agree, n_min, replace=False),
        rng.choice(idx_disagree, n_min, replace=False),
    ])
    rng.shuffle(idx)
    return X[idx], y[idx], z1[idx], z2[idx], groups[idx]


def balance_four_groups(X, y, z1, z2, groups, seed=None):
    """Bal-4G: downsample all four groups to min size. n_sub = 4*n_min."""
    rng = np.random.RandomState(seed)
    n_min = min((groups == g).sum() for g in range(4)) # min of all 4 groups (++, +-, --, -+)
    idx = np.concatenate([
        rng.choice(np.where(groups == g)[0], n_min, replace=False) for g in range(4) # all now size n_min
    ])
    rng.shuffle(idx)
    return X[idx], y[idx], z1[idx], z2[idx], groups[idx]

print('Subsampling functions ready.')

Subsampling functions ready.


### 1.6 Logging & Checkpoint Utilities

In [8]:
# ============================================================
# Cell 8: Save/load utilities for Drive persistence
# ============================================================

def save_csv(df, name):
    """Save DataFrame to Drive CSV directory."""
    path = f'{DRIVE_ROOT}/csv/{name}'
    df.to_csv(path, index=False)
    print(f'  Saved {path} ({len(df)} rows)')


def save_checkpoint(model, metadata, name):
    """Save model weights + metadata dict to Drive."""
    path = f'{DRIVE_ROOT}/checkpoints/{name}.pt'
    torch.save({
        'model_state_dict': model.state_dict(),
        'metadata': metadata,
    }, path)


def log_experiment(exp_name, message):
    """Append a timestamped log line to the experiment log."""
    path = f'{DRIVE_ROOT}/logs/experiment_log.txt'
    ts = time.strftime('%Y-%m-%d %H:%M:%S')
    with open(path, 'a') as f:
        f.write(f'[{ts}] {exp_name}: {message}\n')


def run_single_shortcut_experiment(config, loss_registry=None):
    """Generic runner for a single-shortcut experiment configuration.

    config: dict with keys:
        n, d, rho_train, rho_test, B, seeds, conditions, epochs, lr
    conditions: list of (name, subsample_fn_or_None, subsample_args)

    Returns: list of result dicts
    """
    if loss_registry is None:
        loss_registry = LOSS_FNS_CORE
    results = []
    d = config['d']
    B_val = config['B']

    for seed in config['seeds']:
        # Generate training data
        X_tr, y_tr, z_tr, g_tr = generate_linear_data(
            config['n'], d, config['rho_train'], B_val, seed=seed * 500
        )
        # Fixed test set
        X_te, y_te, z_te, g_te = generate_linear_data(
            N_TEST, d, config['rho_test'], B_val, seed=seed * 500 + 1
        )

        for cond_name, sub_fn, sub_args in config['conditions']:
            # Apply subsampling
            if sub_fn is None:
                Xc, yc, zc, gc = X_tr, y_tr, z_tr, g_tr
            else:
                Xc, yc, zc, gc = sub_fn(X_tr, y_tr, z_tr, g_tr, **sub_args, seed=seed)

            n_sub = len(yc)
            leftover_frac = float(gc.mean())

            for method_name, loss_factory in loss_registry.items():
                torch.manual_seed(seed * 1000 + hash(method_name) % 1000)
                model = LinearModel(d)
                model = train_model(model, Xc, yc, loss_factory(),
                                    epochs=config.get('epochs', EPOCHS),
                                    lr=config.get('lr', LR))

                # Evaluate on test
                test_m = evaluate_single(model, X_te, y_te, g_te, B_val)
                # Evaluate on train (for overfitting analysis)
                train_m = evaluate_single(model, Xc, yc, gc, B_val)

                results.append({
                    'method': method_name,
                    'condition': cond_name,
                    'seed': seed,
                    'n_original': config['n'],
                    'n_train': n_sub,
                    'd': d,
                    'd_over_n': d / n_sub,
                    'B': B_val,
                    'rho_train': config['rho_train'],
                    'rho_test': config['rho_test'],
                    'leftover_frac': leftover_frac,
                    # Test metrics (prefixed)
                    'test_acc': test_m['acc'],
                    'test_wga': test_m['wga'],
                    'test_acc_shortcut': test_m['acc_shortcut'],
                    'test_acc_leftover': test_m['acc_leftover'],
                    'test_margin_gap': test_m['margin_gap'],
                    # Weight analysis
                    'B_wz': test_m['B_wz'],
                    'wy': test_m['wy'],
                    'shortcut_ratio': test_m['shortcut_ratio'],
                    'shortcut_dominates': test_m['shortcut_dominates'],
                    'w_noise_norm': test_m['w_noise_norm'],
                    # Train metrics (for overfitting)
                    'train_acc': train_m['acc'],
                    'train_margin_std': train_m['margin_std'],
                    'gen_gap': train_m['acc'] - test_m['acc'],
                })
    return results

print('Utilities ready. Infrastructure complete.')

Utilities ready. Infrastructure complete.


**Support Vector Analysis**

*   Checking what points are closest to decision boundary



In [ ]:
import matplotlib.pyplot as plt
# helper functions fo r support vector analysis
def get_margins(model, X, y): # function basclly gives us y * f(x) - useful later
    """Compute per-sample margins: y * (w^T x)."""
    model.eval()
    with torch.no_grad():
        X_t = torch.tensor(X, dtype=torch.float32).to(device)
        y_t = torch.tensor(y, dtype=torch.float32).to(device)
        logits = model(X_t).squeeze()
        margins = (y_t * logits).cpu().numpy()
    return margins


def analyze_boundary_samples(X, y, z, groups, margins, k=50):
    """Return stats about the k samples closest to the decision boundary."""
    sorted_idx = np.argsort(np.abs(margins))
    closest_idx = sorted_idx[:k] # k smallest margins

    n_minority = np.sum(groups[closest_idx] == 1)  # how many of leftover group in these k closest points to boundary
    n_majority = np.sum(groups[closest_idx] == 0)  # how many of shortcut group in these k closest points to boundary
    mean_margin = np.mean(np.abs(margins[closest_idx])) # mean margin of k closest points

    return {
        'closest_idx': closest_idx,
        'n_minority_near_boundary': n_minority,
        'n_majority_near_boundary': n_majority,
        'minority_frac_near_boundary': n_minority / k,
        'mean_abs_margin_closest_k': mean_margin,
        'overall_minority_frac': np.mean(groups == 1),
    }


K_CLOSEST = 50  # how many "near boundary" samples to examine
seed = 0

X_full, y_full, z_full, g_full = generate_linear_data(N_TRAIN, D, RHO, B, seed=seed) # generate data

# ── Build conditions ──
conditions = {}
conditions['Full'] = (X_full, y_full, z_full, g_full)

Xgd, ygd, zgd, ggd = group_balanced_downsample(X_full, y_full, z_full, g_full, seed=seed)
conditions['GB-Down'] = (Xgd, ygd, zgd, ggd)

Xgo, ygo, zgo, ggo = group_balanced_oversample(X_full, y_full, z_full, g_full, seed=seed)
conditions['GB-Over'] = (Xgo, ygo, zgo, ggo)

ratio = 0.25
Xrs, yrs, zrs, grs = random_subsample(X_full, y_full, z_full, g_full, ratio=ratio, seed=seed)
conditions[f'Random (r={ratio})'] = (Xrs, yrs, zrs, grs)

In [ ]:
# Train and analyse each condition x loss combination
results = {}
all_margins = {}

for cond_name, (Xc, yc, zc, gc) in conditions.items():
    for method_name, loss_factory in LOSS_FNS.items():
        key = f'{cond_name} | {method_name}'

        torch.manual_seed(seed)
        model = LinearModel(D).to(device)
        model = train_model(model, Xc, yc, loss_factory())

        margins = get_margins(model, Xc, yc)
        stats = analyze_boundary_samples(Xc, yc, zc, gc, margins, k=min(K_CLOSEST, len(yc)))
        stats['n_train'] = len(yc)
        stats['d_over_n'] = D / len(yc)
        stats['condition'] = cond_name
        stats['method'] = method_name
        results[key] = stats
        all_margins[key] = (margins, gc)

        print(f"\n{'='*50}")
        print(f"  {key}  (n={len(yc)}, d/n={D/len(yc):.2f})")
        print(f"  Overall minority fraction:      {stats['overall_minority_frac']:.2f}")
        print(f"  Minority fraction near boundary: {stats['minority_frac_near_boundary']:.2f}")
        print(f"  Mean |margin| of closest {min(K_CLOSEST, len(yc))}: {stats['mean_abs_margin_closest_k']:.4f}")


  Full | ERM  (n=1000, d/n=0.30)
  Overall minority fraction:      0.12
  Minority fraction near boundary: 0.68
  Mean |margin| of closest 50: 8.6987

  Full | sigma-Damp  (n=1000, d/n=0.30)
  Overall minority fraction:      0.12
  Minority fraction near boundary: 0.00
  Mean |margin| of closest 50: 2.5414

  Full | SD  (n=1000, d/n=0.30)
  Overall minority fraction:      0.12
  Minority fraction near boundary: 0.04
  Mean |margin| of closest 50: 1.1719

  Full | Marg-Log  (n=1000, d/n=0.30)
  Overall minority fraction:      0.12
  Minority fraction near boundary: 0.20
  Mean |margin| of closest 50: 0.4676

  GB-Down | ERM  (n=236, d/n=1.27)
  Overall minority fraction:      0.50
  Minority fraction near boundary: 0.48
  Mean |margin| of closest 50: 12.4701

  GB-Down | sigma-Damp  (n=236, d/n=1.27)
  Overall minority fraction:      0.50
  Minority fraction near boundary: 0.52
  Mean |margin| of closest 50: 2.5442

  GB-Down | SD  (n=236, d/n=1.27)
  Overall minority fraction:      0.

Testing on Real Datasets:
- Waterbirds - bird type (waterbird/landbird) is the label, background (water/land) is the spurious shortcut
- SpurBreast - breast MRI classification with exactly **two** identified spurious correlations: magnetic field strength (a global feature influencing the entire image) and image orientation (a local feature affecting spatial alignment)

In [14]:
# modify helper functions for use for real data
def compute_group_metrics(preds, labels, groups, group_names=None):
    """Compute per-group accuracy, sizes, and worst-group accuracy."""
    unique_groups = np.unique(groups)
    results = {}
    for g in unique_groups:
        mask = groups == g
        acc = np.mean(preds[mask] == labels[mask])
        name = group_names[g] if group_names else f'group_{g}'
        results[name] = {
            'accuracy': acc,
            'size': int(np.sum(mask)),
            'fraction': np.mean(mask),
        }
    wga = min(r['accuracy'] for r in results.values())
    avg = np.mean(preds == labels)
    return {'per_group': results, 'wga': wga, 'avg_acc': avg}


def print_group_summary(metrics, dataset_name=''):
    """Pretty-print group metrics."""
    print(f"\n{'='*60}")
    print(f"  {dataset_name} Results")
    print(f"  Average Acc: {metrics['avg_acc']:.4f}  |  WGA: {metrics['wga']:.4f}")
    print(f"{'='*60}")
    for name, info in metrics['per_group'].items():
        print(f"  {name:30s}  n={info['size']:5d} ({info['fraction']:.3f})  acc={info['accuracy']:.4f}")

def real_data_gb_downsample(X, y, groups, seed=0): # strategy adapted for real data as does not take z
    """Group-balanced downsampling: match smallest group size."""
    rng = np.random.RandomState(seed)
    unique_groups = np.unique(groups)
    min_size = min(np.sum(groups == g) for g in unique_groups) # smallest group size in data
    indices = []
    for g in unique_groups:
        g_idx = np.where(groups == g)[0]
        chosen = rng.choice(g_idx, size=min_size, replace=False) # choose min_size samples from g_idx for each group
        indices.extend(chosen)
    indices = np.array(indices)
    rng.shuffle(indices)
    return X[indices], y[indices], groups[indices], indices


def real_data_gb_oversample(X, y, groups, seed=0):
    """Group-balanced oversampling: match largest group size."""
    rng = np.random.RandomState(seed)
    unique_groups = np.unique(groups)
    max_size = max(np.sum(groups == g) for g in unique_groups) # get largest group
    indices = []
    for g in unique_groups:
        g_idx = np.where(groups == g)[0]
        chosen = rng.choice(g_idx, size=max_size, replace=True) # max_size samples for each group
        indices.extend(chosen)
    indices = np.array(indices)
    rng.shuffle(indices)
    return X[indices], y[indices], groups[indices], indices


def real_data_random_subsample(X, y, groups, ratio=0.5, seed=0):
    """Random subsampling preserving group ratios."""
    rng = np.random.RandomState(seed)
    n_keep = int(len(y) * ratio) # fixed amount of data to keep
    indices = rng.choice(len(y), size=n_keep, replace=False)
    return X[indices], y[indices], groups[indices], indices

In [ ]:
# REDEFINE LOSS FUNCTIONS for real data experiments

class LogLoss(nn.Module):
    def forward(self, logits, labels):
        # labels in {0, 1}
        return nn.functional.binary_cross_entropy_with_logits(logits, labels.float())


class SpectralDecoupling(nn.Module):
    """SD loss: log-loss + λ * |f(x)|^2"""
    def __init__(self, lam=0.1):
        super().__init__()
        self.lam = lam

    def forward(self, logits, labels):
        bce = nn.functional.binary_cross_entropy_with_logits(logits, labels.float())
        penalty = self.lam * (logits ** 2).mean()
        return bce + penalty


class MargLogLoss(nn.Module):
    """Marg-Log loss with per-class targets: log-loss + λ * log(1 + |f - γ_y|^2)

    Per-class targets γ_0, γ_1 handle label imbalance.
    """
    def __init__(self, lam=0.1, gamma_0=0.0, gamma_1=1.0):
        super().__init__()
        self.lam = lam
        self.gamma_0 = gamma_0
        self.gamma_1 = gamma_1

    def forward(self, logits, labels):
        bce = nn.functional.binary_cross_entropy_with_logits(logits, labels.float())
        # Per-class targets
        targets = torch.where(labels == 1,
                              torch.tensor(self.gamma_1, device=logits.device),
                              torch.tensor(self.gamma_0, device=logits.device))
        penalty = self.lam * torch.log(1 + (logits.squeeze() - targets) ** 2).mean()
        return bce + penalty


class SigmaDampLoss(nn.Module):
    """σ-Damp loss with per-class temperatures.

    ℓ_σ-damp(y, f) = ℓ_log(T_y * 1.278 * yf * (1 - σ(1.278 * yf)))
    """
    def __init__(self, T_0=1.0, T_1=2.0):
        super().__init__()
        self.T_0 = T_0
        self.T_1 = T_1

    def forward(self, logits, labels):
        # Convert to margin: m = (2*y - 1) * f for y in {0,1}
        signs = 2 * labels.float() - 1  # {-1, +1}
        margins = signs * logits.squeeze()

        # Per-class temperatures
        T = torch.where(labels == 1,
                        torch.tensor(self.T_1, device=logits.device),
                        torch.tensor(self.T_0, device=logits.device))

        # Damped input: T_y * 1.278 * m * (1 - σ(1.278 * m))
        scaled_m = 1.278 * margins
        damped_input = T * scaled_m * (1 - torch.sigmoid(scaled_m))

        # Log-loss on damped input
        loss = torch.log(1 + torch.exp(-damped_input)).mean()
        return loss



In [16]:
# function to load dataset from saved location in drive
from torch.utils.data.dataset import Dataset
from torchvision import transforms, models

def load_waterbirds(root_dir='/content/drive/MyDrive'): # loads dataset
    """Load Waterbirds dataset from WILDS. Returns embeddings + labels + groups."""
    from wilds import get_dataset
    dataset = get_dataset(dataset='waterbirds', root_dir=root_dir, download=False)
    resnet = models.resnet50(pretrained=True)
    resnet.fc = nn.Identity() # get rid of last layer in resente so we only use embeddings
    resnet.eval().to(device)
    transform = transforms.Compose([
        transforms.Resize((224, 224)), # resize for resnet
        transforms.ToTensor()
    ])
    splits = {}
    for split_name in ['train', 'test']:
        subset = dataset.get_subset(split_name, transform=transform)
        loader = torch.utils.data.DataLoader(subset, batch_size=64, num_workers=2)
        all_features, all_labels, all_groups = [], [], []
        with torch.no_grad():
            for batch in loader:
                x, y_true, metadata = batch
                features = resnet(x.to(device)).cpu().numpy()
                all_features.append(features)
                all_labels.append(y_true.numpy())
                place = metadata[:, 0].numpy()
                groups = 2 * y_true.numpy() + place
                all_groups.append(groups)
        splits[split_name] = {
            'X': np.concatenate(all_features).astype(np.float32),
            'y': np.concatenate(all_labels).astype(np.float32), # label is now between 0 - 1
            #'y': (np.concatenate(all_labels).astype(float) * 2 - 1).astype(np.float32),
            'groups': np.concatenate(all_groups).astype(int),
        }
    group_names = {0: 'landbird_land', 1: 'landbird_water', 2: 'waterbird_land', 3: 'waterbird_water'} # create groups for identifying shortcuts
    return splits, group_names

In [9]:
# install for waterbvirds dataset
!pip install wilds

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 126.2/126.2 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.8/78.8 kB 10.4 MB/s eta 0:00:00


In [ ]:
# making sure waterbirds are in correct location
# Extract the contents directly into the waterbirds_v1.0 directory
!tar -xf /content/drive/MyDrive/waterbirds_v1.0/archive.tar.gz -C /content/drive/MyDrive/waterbirds_v1.0/

# Verify that metadata.csv is now in the correct spot!
!ls -l /content/drive/MyDrive/waterbirds_v1.0/metadata.csv

In [17]:
# Waterbirds Hyperparameter Tuning LINEAR PROBE  # ACTUAL

# change SD to make more stable for waterbirds dataset: Uses softplus and manual margin conversion
class SpectralDecoupling(nn.Module):
    """Spectral Decoupling: log-loss + lambda * ||f||^2."""
    def __init__(self, lam=0.1):
        super().__init__()
        self.lam = lam # param
    def forward(self, logits, labels):
        f = logits.squeeze()
        # Convert {0, 1} to {-1, 1} locally
        targets = 2 * labels.float() - 1
        # Use softplus for better numerical stability than log(1 + exp(-x))
        loss = torch.nn.functional.softplus(-targets * f).mean()
        return loss + self.lam * (f ** 2).mean()

WATERBIRDS = True
if WATERBIRDS:
    splits, group_names = load_waterbirds()
    X_tr = splits['train']['X']
    y_tr = splits['train']['y']
    X_te = splits['test']['X']
    y_te = splits['test']['y']
    g_te = splits['test']['groups']

    configs = [
        ('SD_lam005',  SpectralDecoupling(lam=0.05)),
        ('SD_lam01',   SpectralDecoupling(lam=0.1)),
        ('SD_lam05',   SpectralDecoupling(lam=0.5)),
        ('ML_lam001_g2',  MargLogLoss(lam=0.01, gamma_0=0, gamma_1=2)),
        ('ML_lam005_g2',  MargLogLoss(lam=0.05, gamma_0=0, gamma_1=2)),
        ('ML_lam01_g2',   MargLogLoss(lam=0.1,  gamma_0=0, gamma_1=2)),
        ('ML_lam05_g2',   MargLogLoss(lam=0.5,  gamma_0=0, gamma_1=2)),
        ('ML_lam1_g2',    MargLogLoss(lam=1.0,  gamma_0=0, gamma_1=2)),
        ('ML_g1_05',   MargLogLoss(lam=0.1, gamma_0=0, gamma_1=0.5)),
        ('ML_g1_1',    MargLogLoss(lam=0.1, gamma_0=0, gamma_1=1)),
        ('ML_g1_2',    MargLogLoss(lam=0.1, gamma_0=0, gamma_1=2)),
        ('σD_T1_05',  SigmaDampLoss(T_0=1, T_1=0.5)),
        ('σD_T1_1',   SigmaDampLoss(T_0=1, T_1=1)),
        ('σD_T1_2',   SigmaDampLoss(T_0=1, T_1=2)),
        ('σD_T1_3',   SigmaDampLoss(T_0=1, T_1=3)),
        ('σD_T0_05',  SigmaDampLoss(T_0=0.5, T_1=2)),
        ('σD_T0_1',   SigmaDampLoss(T_0=1,   T_1=2)),
        ('σD_T0_2',   SigmaDampLoss(T_0=2,   T_1=2)),
    ]
    d = X_tr.shape[1]
    hp_results = []

    for config_name, loss_factory in configs:
        print(f"\n{'='*60}")
        print(f"  Waterbirds HP: {config_name}")
        print(f"{'='*60}")

        torch.manual_seed(0)
        model = nn.Linear(d, 1, bias=False).to(device)
        optimizer = optim.Adam(model.parameters(), lr=0.01)

        Xt = torch.tensor(X_tr, dtype=torch.float32).to(device)
        yt = torch.tensor(y_tr, dtype=torch.float32).to(device)

        loss_fn = loss_factory
        for epoch in range(3000):
            optimizer.zero_grad()
            logits = model(Xt).squeeze()
            loss = loss_fn(logits, yt)
            loss.backward()
            optimizer.step()

        model.eval()
        with torch.no_grad():
            test_logits = model(torch.tensor(X_te, dtype=torch.float32).to(device)).squeeze().cpu().numpy()
            # test_preds = np.sign(test_logits)
            # test_preds[test_preds == 0] = 1
            test_preds = (test_logits > 0).astype(float)

        metrics = compute_group_metrics(test_preds, y_te, g_te, group_names)
        print(f"  => avg_acc={metrics['avg_acc']:.3f} | wga={metrics['wga']:.3f}")

        hp_results.append({
            'config': config_name,
            'avg_acc': metrics['avg_acc'],
            'wga': metrics['wga'],
        })

    df_hp_wb = pd.DataFrame(hp_results)
    save_csv(df_hp_wb, 'results_waterbirds_hyperparam.csv')
    print(df_hp_wb.to_string(index=False, float_format='{:.3f}'.format))


  Waterbirds HP: SD_lam005
  => avg_acc=0.726 | wga=0.466

  Waterbirds HP: SD_lam01
  => avg_acc=0.837 | wga=0.254

  Waterbirds HP: SD_lam05
  => avg_acc=0.843 | wga=0.104

  Waterbirds HP: ML_lam001_g2
  => avg_acc=0.811 | wga=0.491

  Waterbirds HP: ML_lam005_g2
  => avg_acc=0.798 | wga=0.456

  Waterbirds HP: ML_lam01_g2
  => avg_acc=0.780 | wga=0.486

  Waterbirds HP: ML_lam05_g2
  => avg_acc=0.471 | wga=0.064

  Waterbirds HP: ML_lam1_g2
  => avg_acc=0.640 | wga=0.255

  Waterbirds HP: ML_g1_05
  => avg_acc=0.805 | wga=0.430

  Waterbirds HP: ML_g1_1
  => avg_acc=0.800 | wga=0.430

  Waterbirds HP: ML_g1_2
  => avg_acc=0.780 | wga=0.486

  Waterbirds HP: σD_T1_05
  => avg_acc=0.823 | wga=0.393

  Waterbirds HP: σD_T1_1
  => avg_acc=0.705 | wga=0.310

  Waterbirds HP: σD_T1_2
  => avg_acc=0.785 | wga=0.449

  Waterbirds HP: σD_T1_3
  => avg_acc=0.777 | wga=0.456

  Waterbirds HP: σD_T0_05
  => avg_acc=0.770 | wga=0.480

  Waterbirds HP: σD_T0_1
  => avg_acc=0.785 | wga=0.449

  

In [10]:

def train_finetune_one_epoch(model, loader, optimizer, loss_fn, device):
    """Train one epoch of full finetuning."""
    model.train()
    total_loss = 0
    n_samples = 0
    for x, y, metadata in loader:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        logits = model(x).squeeze(-1)
        loss = loss_fn(logits, y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * x.size(0)
        n_samples += x.size(0)
    return total_loss / n_samples


def evaluate_finetune(model, loader, device):
    """Evaluate with per-group accuracy."""
    model.eval()
    all_preds, all_labels, all_groups = [], [], []
    with torch.no_grad():
        for x, y, metadata in loader:
            x = x.to(device)
            logits = model(x).squeeze(-1)
            preds = (logits > 0).long().cpu()
            all_preds.append(preds)
            all_labels.append(y)
            # metadata[:, 0] is the background in waterbirds
            groups = 2 * y + metadata[:, 0]  # 4 groups
            all_groups.append(groups)

    preds = torch.cat(all_preds)
    labels = torch.cat(all_labels)
    groups = torch.cat(all_groups)

    correct = (preds == labels).float()
    group_accs = {}
    for g in groups.unique():
        mask = groups == g
        group_accs[g.item()] = correct[mask].mean().item()

    avg_acc = correct.mean().item()
    wga = min(group_accs.values()) if group_accs else 0.0
    return avg_acc, wga, group_accs


def reproduce_puli_waterbirds(data_dir='./data', device='cuda'):
    """
    Reproduce Puli et al. Table 1 Waterbirds results.
    Using single robust configurations.
    """
    import torchvision.models as models

    dataset, splits = load_waterbirds_wilds(data_dir)

    # Streamlined robust configuration
    hp_grid = {
        'lr': [1e-5],
        'wd': [0.1],
    }

    # Single best-guess method-specific hyperparameters
    method_configs = {
        'ERM': [{'loss_fn': LogLoss()}],
        'SD': [{'loss_fn': SpectralDecoupling(lam=0.1), 'gamma_0': 0, 'gamma_1': 2}],
        'Marg-Log': [{'loss_fn': MargLogLoss(lam=0.1, gamma_0=0, gamma_1=2)}],
        'σ-damp': [{'loss_fn': SigmaDampLoss(T_0=1, T_1=2)}],
    }

    train_loader = DataLoader(splits['train'], batch_size=128, shuffle=True, num_workers=2)
    val_loader = DataLoader(splits['val'], batch_size=128, shuffle=False, num_workers=2)
    test_loader = DataLoader(splits['test'], batch_size=128, shuffle=False, num_workers=2)

    results = {}

    for method_name, configs in method_configs.items():
        best_val_wga = -1
        best_test_result = None

        for config in configs:
            for lr in hp_grid['lr']:
                for wd in hp_grid['wd']:
                    # Fresh model each run
                    model = models.resnet50(pretrained=True)
                    model.fc = nn.Linear(2048, 1)
                    model = model.to(device)

                    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=wd)
                    loss_fn = config['loss_fn']

                    best_epoch_val_wga = -1
                    best_epoch_model = None

                    for epoch in range(100):
                        train_finetune_one_epoch(model, train_loader, optimizer, loss_fn, device)

                        # Evaluate on validation set
                        _, val_wga, _ = evaluate_finetune(model, val_loader, device)

                        if val_wga > best_epoch_val_wga:
                            best_epoch_val_wga = val_wga
                            best_epoch_model = copy.deepcopy(model.state_dict())

                    if best_epoch_val_wga > best_val_wga:
                        best_val_wga = best_epoch_val_wga
                        # Evaluate best model on test
                        model.load_state_dict(best_epoch_model)
                        test_acc, test_wga, test_groups = evaluate_finetune(model, test_loader, device)
                        best_test_result = {
                            'method': method_name,
                            'lr': lr, 'wd': wd,
                            'val_wga': best_val_wga,
                            'test_wga': test_wga,
                            'test_acc': test_acc,
                            'test_groups': test_groups,
                        }
                        print(f"  {method_name} | lr={lr} wd={wd} | val_wga={best_val_wga:.3f} test_wga={test_wga:.3f}")

        results[method_name] = best_test_result
        print(f"\n{'='*60}")
        print(f"  BEST {method_name}: test_wga={best_test_result['test_wga']:.3f}")
        print(f"{'='*60}\n")

    return results


# ============================================================================
# PART 2: Fix Linear Probe Experiments
# ============================================================================
def load_waterbirds_wilds(data_dir='./data'):
    """Load Waterbirds via WILDS with standard ImageNet transforms."""
    from wilds import get_dataset
    from torchvision import transforms

    # Standard ImageNet preprocessing required for ResNet-50
    transform = transforms.Compose([
        transforms.Resize(256),
        transforms.CenterCrop(224),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406],
                             std=[0.229, 0.224, 0.225])
    ])

    # download=False because we already extracted it manually
    dataset = get_dataset(dataset='waterbirds', root_dir=data_dir, download=False)

    splits = {}
    for split_name in ['train', 'val', 'test']:
        # Apply the transform to each subset!
        split_data = dataset.get_subset(split_name, transform=transform)
        splits[split_name] = split_data

    return dataset, splits


def extract_embeddings(data_dir='./data', device='cuda'):
    """Extract ResNet-50 embeddings for Waterbirds."""
    import torchvision.models as models

    dataset, splits_raw = load_waterbirds_wilds(data_dir)

    # Updated weights argument to fix the deprecation warning
    backbone = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)
    backbone.fc = nn.Identity()
    backbone = backbone.to(device)
    backbone.eval()

    embeddings = {}
    for split_name in ['train', 'val', 'test']:
        split_data = splits_raw[split_name]
        # Lowered num_workers to 2 to fix the PyTorch warning/freezing
        loader = DataLoader(split_data, batch_size=64, shuffle=False, num_workers=2)

        all_X, all_y, all_groups = [], [], []
        with torch.no_grad():
            for x, y, metadata in loader:
                x = x.to(device)
                feats = backbone(x).cpu().numpy()
                all_X.append(feats)
                all_y.append(y.numpy())
                groups = 2 * y + metadata[:, 0]
                all_groups.append(groups.numpy())

        embeddings[split_name] = {
            'X': np.concatenate(all_X),
            'y': np.concatenate(all_y),
            'groups': np.concatenate(all_groups),
        }

    return embeddings


def run_linear_probe_fixed(X_train, y_train, groups_train,
                           X_val, y_val, groups_val,
                           X_test, y_test, groups_test,
                           loss_fn, lr=0.01, epochs=3000,
                           device='cuda', use_bias=True):
    """
    Train linear probe with validation-based early stopping.
    """
    d = X_train.shape[1]

    X_tr = torch.tensor(X_train, dtype=torch.float32, device=device)
    y_tr = torch.tensor(y_train, dtype=torch.long, device=device)
    g_tr = torch.tensor(groups_train, dtype=torch.long, device=device)

    X_v = torch.tensor(X_val, dtype=torch.float32, device=device)
    y_v = torch.tensor(y_val, dtype=torch.long, device=device)
    g_v = torch.tensor(groups_val, dtype=torch.long, device=device)

    X_te = torch.tensor(X_test, dtype=torch.float32, device=device)
    y_te = torch.tensor(y_test, dtype=torch.long, device=device)
    g_te = torch.tensor(groups_test, dtype=torch.long, device=device)

    model = nn.Linear(d, 1, bias=use_bias).to(device)
    optimizer = optim.Adam(model.parameters(), lr=lr)

    best_val_wga = -1
    best_state = None
    patience = 500
    patience_counter = 0

    for epoch in range(epochs):
        model.train()
        optimizer.zero_grad()
        logits = model(X_tr).squeeze(-1)
        loss = loss_fn(logits, y_tr)
        loss.backward()
        optimizer.step()

        if (epoch + 1) % 50 == 0:
            model.eval()
            with torch.no_grad():
                val_logits = model(X_v).squeeze(-1)
                val_preds = (val_logits > 0).long()
                val_correct = (val_preds == y_v).float()

                val_group_accs = []
                for g in g_v.unique():
                    mask = g_v == g
                    if mask.sum() > 0:
                        val_group_accs.append(val_correct[mask].mean().item())

                val_wga = min(val_group_accs) if val_group_accs else 0.0

                if val_wga > best_val_wga:
                    best_val_wga = val_wga
                    best_state = copy.deepcopy(model.state_dict())
                    patience_counter = 0
                else:
                    patience_counter += 50

                if patience_counter >= patience:
                    break

    if best_state is not None:
        model.load_state_dict(best_state)

    model.eval()
    with torch.no_grad():
        test_logits = model(X_te).squeeze(-1)
        test_preds = (test_logits > 0).long()
        test_correct = (test_preds == y_te).float()

        test_avg = test_correct.mean().item()
        test_group_accs = {}
        for g in g_te.unique():
            mask = g_te == g
            test_group_accs[g.item()] = test_correct[mask].mean().item()
        test_wga = min(test_group_accs.values())

        train_logits = model(X_tr).squeeze(-1)
        train_loss = loss_fn(train_logits, y_tr).item()
        train_preds = (train_logits > 0).long()
        train_acc = (train_preds == y_tr).float().mean().item()

    return {
        'test_wga': test_wga,
        'test_avg': test_avg,
        'test_groups': test_group_accs,
        'val_wga': best_val_wga,
        'train_loss': train_loss,
        'train_acc': train_acc,
    }


def run_fixed_linear_probe_experiment(embeddings, device='cuda'):
    """
    Run the FIXED linear probe experiment with single robust hyperparameters.
    """
    X_tr = embeddings['train']['X']
    y_tr = embeddings['train']['y']
    g_tr = embeddings['train']['groups']
    X_val = embeddings['val']['X']
    y_val = embeddings['val']['y']
    g_val = embeddings['val']['groups']
    X_te = embeddings['test']['X']
    y_te = embeddings['test']['y']
    g_te = embeddings['test']['groups']

    from sklearn.preprocessing import StandardScaler
    scaler = StandardScaler()
    X_tr = scaler.fit_transform(X_tr).astype(np.float32)
    X_val = scaler.transform(X_val).astype(np.float32)
    X_te = scaler.transform(X_te).astype(np.float32)

    print(f"Data: train={len(y_tr)}, val={len(y_val)}, test={len(y_te)}, d={X_tr.shape[1]}")
    print(f"d/n_train = {X_tr.shape[1] / len(y_tr):.3f}")

    # Single robust learning rate
    lr_grid = [0.01]

    # ---- Method configs with single per-class hyperparameters ----
    method_configs = {
        'ERM': [{'loss_fn': LogLoss(), 'name': 'ERM'}],
        'SD': [{'loss_fn': SpectralDecoupling(lam=0.1), 'name': 'SD(λ=0.1)'}],
        'Marg-Log': [{'loss_fn': MargLogLoss(lam=0.1, gamma_0=0, gamma_1=2), 'name': 'ML(γ0=0,γ1=2)'}],
        'σ-damp': [{'loss_fn': SigmaDampLoss(T_0=1, T_1=2), 'name': 'σD(T0=1,T1=2)'}],
    }

    all_results = []

    for method_name, configs in method_configs.items():
        best_val_wga = -1
        best_result = None

        for config in configs:
            for lr in lr_grid:
                for seed in range(3):
                    torch.manual_seed(seed)
                    np.random.seed(seed)

                    result = run_linear_probe_fixed(
                        X_tr, y_tr, g_tr,
                        X_val, y_val, g_val,
                        X_te, y_te, g_te,
                        loss_fn=config['loss_fn'],
                        lr=lr, epochs=5000,
                        device=device, use_bias=True
                    )
                    result['method'] = method_name
                    result['config'] = config['name']
                    result['lr'] = lr
                    result['seed'] = seed
                    all_results.append(result)

                    if result['val_wga'] > best_val_wga:
                        best_val_wga = result['val_wga']
                        best_result = result

        print(f"\n{'='*60}")
        print(f"  BEST {method_name}: val_wga={best_result['val_wga']:.3f} "
              f"test_wga={best_result['test_wga']:.3f}")
        print(f"  Config: {best_result['config']}, lr={best_result['lr']}")
        print(f"  Per-group: {best_result['test_groups']}")
        print(f"{'='*60}")

    return pd.DataFrame(all_results)


def run_fixed_with_subsampling(embeddings, device='cuda'):
    """
    Run the fixed linear probe with subsampling strategies.
    """
    X_tr = embeddings['train']['X']
    y_tr = embeddings['train']['y']
    g_tr = embeddings['train']['groups']
    X_val = embeddings['val']['X']
    y_val = embeddings['val']['y']
    g_val = embeddings['val']['groups']
    X_te = embeddings['test']['X']
    y_te = embeddings['test']['y']
    g_te = embeddings['test']['groups']

    from sklearn.preprocessing import StandardScaler
    scaler = StandardScaler()
    X_tr_scaled = scaler.fit_transform(X_tr).astype(np.float32)
    X_val_scaled = scaler.transform(X_val).astype(np.float32)
    X_te_scaled = scaler.transform(X_te).astype(np.float32)

    # Best hyperparams per method locked in
    methods = {
        'ERM': LogLoss(),
        'SD': SpectralDecoupling(lam=0.1),
        'Marg-Log': MargLogLoss(lam=0.1, gamma_0=0, gamma_1=2),
        'σ-damp': SigmaDampLoss(T_0=1, T_1=2),
    }

    conditions = ['Full', 'GB-Down', 'GB-Over', 'Random (r=0.25)']
    all_results = []

    for cond in conditions:
        if cond == 'Full':
            X_c, y_c, g_c = X_tr_scaled, y_tr, g_tr
        elif cond == 'GB-Down':
            X_c, y_c, g_c = gb_downsample(X_tr_scaled, y_tr, g_tr)
        elif cond == 'GB-Over':
            X_c, y_c, g_c = gb_oversample(X_tr_scaled, y_tr, g_tr)
        elif cond == 'Random (r=0.25)':
            idx = np.random.choice(len(y_tr), size=len(y_tr)//4, replace=False)
            X_c, y_c, g_c = X_tr_scaled[idx], y_tr[idx], g_tr[idx]

        print(f"\n{'='*60}")
        print(f"  {cond}: n={len(y_c)}, d/n={X_c.shape[1]/len(y_c):.3f}")
        print(f"{'='*60}")

        for method_name, loss_fn in methods.items():
            seed_results = []
            for seed in range(3):
                torch.manual_seed(seed)
                np.random.seed(seed)

                result = run_linear_probe_fixed(
                    X_c, y_c, g_c,
                    X_val_scaled, y_val, g_val,
                    X_te_scaled, y_te, g_te,
                    loss_fn=loss_fn, lr=0.01, epochs=5000,
                    device=device, use_bias=True
                )
                seed_results.append(result['test_wga'])

                all_results.append({
                    'condition': cond,
                    'method': method_name,
                    'seed': seed,
                    'test_wga': result['test_wga'],
                    'val_wga': result['val_wga'],
                    'd_over_n': X_c.shape[1] / len(y_c),
                    'n_train': len(y_c),
                })

            mean_wga = np.mean(seed_results)
            std_wga = np.std(seed_results)
            print(f"  {method_name:>10s} | WGA = {mean_wga:.3f} ± {std_wga:.3f}")

    return pd.DataFrame(all_results)


def gb_downsample(X, y, groups):
    """Group-balanced downsampling: downsample each group to min group size."""
    unique_groups = np.unique(groups)
    min_size = min(np.sum(groups == g) for g in unique_groups)

    indices = []
    for g in unique_groups:
        g_idx = np.where(groups == g)[0]
        chosen = np.random.choice(g_idx, size=min_size, replace=False)
        indices.extend(chosen)

    indices = np.array(indices)
    return X[indices], y[indices], groups[indices]


def gb_oversample(X, y, groups):
    """Group-balanced oversampling: oversample each group to max group size."""
    unique_groups = np.unique(groups)
    max_size = max(np.sum(groups == g) for g in unique_groups)

    indices = []
    for g in unique_groups:
        g_idx = np.where(groups == g)[0]
        if len(g_idx) < max_size:
            chosen = np.random.choice(g_idx, size=max_size, replace=True)
        else:
            chosen = g_idx
        indices.extend(chosen)

    indices = np.array(indices)
    return X[indices], y[indices], groups[indices]


#PCA + Fixed Linear Probe

def run_pca_fixed(embeddings, d_values=[32, 64, 128, 256, 512], device='cuda'):
    """
    Run PCA experiment with PROPER per-class hyperparameters and validation.
    """
    from sklearn.decomposition import PCA
    from sklearn.preprocessing import StandardScaler

    X_tr = embeddings['train']['X']
    y_tr = embeddings['train']['y']
    g_tr = embeddings['train']['groups']
    X_val = embeddings['val']['X']
    y_val = embeddings['val']['y']
    g_val = embeddings['val']['groups']
    X_te = embeddings['test']['X']
    y_te = embeddings['test']['y']
    g_te = embeddings['test']['groups']

    scaler = StandardScaler()
    X_tr_s = scaler.fit_transform(X_tr)
    X_val_s = scaler.transform(X_val)
    X_te_s = scaler.transform(X_te)

    all_results = []

    for d in d_values:
        print(f"\n{'='*60}")
        print(f"  PCA d={d}")
        print(f"{'='*60}")

        pca = PCA(n_components=d)
        X_tr_pca = pca.fit_transform(X_tr_s).astype(np.float32)
        X_val_pca = pca.transform(X_val_s).astype(np.float32)
        X_te_pca = pca.transform(X_te_s).astype(np.float32)

        var_explained = pca.explained_variance_ratio_.sum()
        print(f"  Variance explained: {var_explained:.3f}")
        print(f"  d/n = {d / len(y_tr):.4f}")

        shortcut = ((g_tr == 1) | (g_tr == 3)).astype(float)
        corrs = [np.abs(np.corrcoef(X_tr_pca[:, i], shortcut)[0, 1]) for i in range(min(5, d))]
        print(f"  Top 5 PC-shortcut correlations: {[f'{c:.3f}' for c in corrs]}")

        # Test each method with locked hyperparameters
        methods = {
            'ERM': LogLoss(),
            'SD': SpectralDecoupling(lam=0.1),
            'Marg-Log': MargLogLoss(lam=0.1, gamma_0=0, gamma_1=2),
            'σ-damp': SigmaDampLoss(T_0=1, T_1=2),
        }

        for method_name, loss_fn in methods.items():
            seed_results = []
            for seed in range(3):
                torch.manual_seed(seed)
                np.random.seed(seed)
                result = run_linear_probe_fixed(
                    X_tr_pca, y_tr, g_tr,
                    X_val_pca, y_val, g_val,
                    X_te_pca, y_te, g_te,
                    loss_fn=loss_fn, lr=0.01, epochs=5000,
                    device=device, use_bias=True
                )
                seed_results.append(result['test_wga'])
                all_results.append({
                    'pca_d': d,
                    'method': method_name,
                    'seed': seed,
                    'test_wga': result['test_wga'],
                    'val_wga': result['val_wga'],
                    'var_explained': var_explained,
                })

            mean_wga = np.mean(seed_results)
            print(f"  {method_name:>10s} | WGA = {mean_wga:.3f}")

    return pd.DataFrame(all_results)
def run_dn_ratio_experiment(embeddings, device='cuda'):
    """
    Sweep over different subsampling fractions to explicitly test the effect of the d/n ratio.
    As n decreases, d/n increases, pushing the model further into the overparameterized regime
    where ERM is theoretically proven to rely more on the shortcut.
    """
    X_tr = embeddings['train']['X']
    y_tr = embeddings['train']['y']
    g_tr = embeddings['train']['groups']
    X_val = embeddings['val']['X']
    y_val = embeddings['val']['y']
    g_val = embeddings['val']['groups']
    X_te = embeddings['test']['X']
    y_te = embeddings['test']['y']
    g_te = embeddings['test']['groups']

    from sklearn.preprocessing import StandardScaler
    scaler = StandardScaler()
    X_tr_scaled = scaler.fit_transform(X_tr).astype(np.float32)
    X_val_scaled = scaler.transform(X_val).astype(np.float32)
    X_te_scaled = scaler.transform(X_te).astype(np.float32)

    # Locked hyperparameters
    methods = {
        'ERM': LogLoss(),
        'SD': SpectralDecoupling(lam=0.1),
        'Marg-Log': MargLogLoss(lam=0.1, gamma_0=0, gamma_1=2),
        'σ-damp': SigmaDampLoss(T_0=1, T_1=2),
    }

    # Fractions of the training data to keep
    fractions = [1.0, 0.5, 0.25, 0.1, 0.05]
    all_results = []

    for frac in fractions:
        n_samples = int(len(y_tr) * frac)
        d = X_tr_scaled.shape[1]
        d_over_n = d / n_samples

        print(f"\n{'='*60}")
        print(f"  Subsampling Fraction: {frac} | n={n_samples} | d/n={d_over_n:.3f}")
        print(f"{'='*60}")

        for method_name, loss_fn in methods.items():
            seed_results = []
            for seed in range(3):
                # Randomly subsample for this seed
                np.random.seed(seed)
                torch.manual_seed(seed)

                idx = np.random.choice(len(y_tr), size=n_samples, replace=False)
                X_c, y_c, g_c = X_tr_scaled[idx], y_tr[idx], g_tr[idx]

                result = run_linear_probe_fixed(
                    X_c, y_c, g_c,
                    X_val_scaled, y_val, g_val,
                    X_te_scaled, y_te, g_te,
                    loss_fn=loss_fn, lr=0.01, epochs=5000,
                    device=device, use_bias=True
                )
                seed_results.append(result['test_wga'])

                all_results.append({
                    'fraction': frac,
                    'n_train': n_samples,
                    'd_over_n': d_over_n,
                    'method': method_name,
                    'seed': seed,
                    'test_wga': result['test_wga'],
                    'val_wga': result['val_wga'],
                })

            mean_wga = np.mean(seed_results)
            std_wga = np.std(seed_results)
            print(f"  {method_name:>10s} | WGA = {mean_wga:.3f} ± {std_wga:.3f}")

    return pd.DataFrame(all_results)

def get_balanced_subset(X, y, groups, k_per_group):
    """
    Generalized group-balanced sampler.
    If a group has more than k_per_group, it downsamples.
    If a group has fewer than k_per_group, it oversamples (with replacement).
    """
    unique_groups = np.unique(groups)
    indices = []

    for g in unique_groups:
        g_idx = np.where(groups == g)[0]
        if len(g_idx) >= k_per_group:
            chosen = np.random.choice(g_idx, size=k_per_group, replace=False)
        else:
            # Oversample minority groups to reach k_per_group
            chosen = np.random.choice(g_idx, size=k_per_group, replace=True)
        indices.extend(chosen)

    indices = np.array(indices)
    # Shuffle the resulting dataset
    shuffle_idx = np.random.permutation(len(indices))
    return X[indices[shuffle_idx]], y[indices[shuffle_idx]], groups[indices[shuffle_idx]]

def run_dn_balanced_sweep(embeddings, device='cuda'):
    """
    Sweeps through different dataset sizes (N) to vary the d/n ratio.
    At each N, it compares purely Random subsampling vs Group-Balanced sampling.
    """
    X_tr = embeddings['train']['X']
    y_tr = embeddings['train']['y']
    g_tr = embeddings['train']['groups']
    X_val = embeddings['val']['X']
    y_val = embeddings['val']['y']
    g_val = embeddings['val']['groups']
    X_te = embeddings['test']['X']
    y_te = embeddings['test']['y']
    g_te = embeddings['test']['groups']

    from sklearn.preprocessing import StandardScaler
    scaler = StandardScaler()
    X_tr_scaled = scaler.fit_transform(X_tr).astype(np.float32)
    X_val_scaled = scaler.transform(X_val).astype(np.float32)
    X_te_scaled = scaler.transform(X_te).astype(np.float32)

    # Calculate exact targets for GB-Down and GB-Over based on training data
    unique_groups, group_counts = np.unique(g_tr, return_counts=True)
    min_g = int(np.min(group_counts))
    max_g = int(np.max(group_counts))

    # Define our sweep targets for K (samples per group)
    # This covers extreme overparameterization (K=25) up to full GB-Over (K=max_g)
    k_targets = {
        'Extreme Small': 25,
        'GB-Down Exact': min_g,
        'Medium Balanced': 500,
        'Full Data Equivalent': len(y_tr) // 4,
        'GB-Over Exact': max_g
    }

    methods = {
        'ERM': LogLoss(),
        'SD': SpectralDecoupling(lam=0.05),
        'Marg-Log': MargLogLoss(lam=0.01, gamma_0=0, gamma_1=2),
        'σ-damp': SigmaDampLoss(T_0=0, T_1=0.5),
    }

    all_results = []
    d = X_tr_scaled.shape[1]

    for label, k in k_targets.items():
        n_total = k * 4
        d_over_n = d / n_total

        print(f"\n{'='*70}")
        print(f"  Target: {label} | Total N={n_total} (K={k}/group) | d/n = {d_over_n:.3f}")
        print(f"{'='*70}")

        for sampling_type in ['Random', 'Balanced']:
            print(f"\n  --- Sampling: {sampling_type} ---")

            for method_name, loss_fn in methods.items():
                seed_results = []
                for seed in range(3):
                    np.random.seed(seed)
                    torch.manual_seed(seed)

                    if sampling_type == 'Balanced':
                        X_c, y_c, g_c = get_balanced_subset(X_tr_scaled, y_tr, g_tr, k)
                    else:
                        # Randomly sample n_total points (allowing replacement if n_total > len(y_tr) for GB-Over equivalent)
                        replace = n_total > len(y_tr)
                        idx = np.random.choice(len(y_tr), size=n_total, replace=replace)
                        X_c, y_c, g_c = X_tr_scaled[idx], y_tr[idx], g_tr[idx]

                    result = run_linear_probe_fixed(
                        X_c, y_c, g_c,
                        X_val_scaled, y_val, g_val,
                        X_te_scaled, y_te, g_te,
                        loss_fn=loss_fn, lr=0.01, epochs=5000,
                        device=device, use_bias=True
                    )
                    seed_results.append(result['test_wga'])

                    all_results.append({
                        'target_label': label,
                        'sampling_type': sampling_type,
                        'n_total': n_total,
                        'd_over_n': d_over_n,
                        'method': method_name,
                        'seed': seed,
                        'test_wga': result['test_wga'],
                        'val_wga': result['val_wga'],
                    })

                mean_wga = np.mean(seed_results)
                std_wga = np.std(seed_results)
                print(f"    {method_name:>10s} | WGA = {mean_wga:.3f} ± {std_wga:.3f}")

    return pd.DataFrame(all_results)

In [ ]:
# RUN LINEAR PROBE TESTING WATERBIRDS
# Set parameters manually
MODE = 'fix_linear'  # Options: 'reproduce', 'fix_linear', 'fix_subsample', 'pca'
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

if MODE == 'fix_linear':
    print("Extracting embeddings...")
    embeddings = extract_embeddings(DATA_DIR, DEVICE)
    print("Running Fixed Linear Probe Experiment...")
    df = run_fixed_linear_probe_experiment(embeddings, DEVICE)
    df.to_csv('results_waterbirds_fixed_linear.csv', index=False)

elif MODE == 'fix_subsample':
    embeddings = extract_embeddings(DATA_DIR, DEVICE)
    df = run_fixed_with_subsampling(embeddings, DEVICE)
    df.to_csv('results_waterbirds_fixed_subsample.csv', index=False)

In [13]:
## WATERBIRDS HYPERPARAMETER FINETUNING # ACTUAL

# ── Waterbirds HP Tuning (Finetuning) ──
# Uses reproduce_puli_waterbirds style but sweeps hyperparameters

from wilds import get_dataset
import torchvision.models as models

DATA_DIR = '/content/drive/MyDrive'
dataset, splits = load_waterbirds_wilds(DATA_DIR)

train_loader = DataLoader(splits['train'], batch_size=64, shuffle=True, num_workers=4)
val_loader = DataLoader(splits['val'], batch_size=64, shuffle=False, num_workers=4)
test_loader = DataLoader(splits['test'], batch_size=64, shuffle=False, num_workers=4)

configs = [
    ('SD_lam005',  SpectralDecoupling(lam=0.05)),
    ('SD_lam01',   SpectralDecoupling(lam=0.1)),
    ('SD_lam05',   SpectralDecoupling(lam=0.5)),
    ('ML_lam01',   MargLogLoss(lam=0.1, gamma_0=0, gamma_1=2)),
    ('ML_lam05',   MargLogLoss(lam=0.5, gamma_0=0, gamma_1=2)),
    ('ML_lam1',    MargLogLoss(lam=1.0, gamma_0=0, gamma_1=2)),
    ('ML_g1_1',    MargLogLoss(lam=0.1, gamma_0=0, gamma_1=1)),
    ('ML_g1_4',    MargLogLoss(lam=0.1, gamma_0=0, gamma_1=4)),
    ('σD_T05',     SigmaDampLoss(T_0=0.5, T_1=2)),
    ('σD_T1',      SigmaDampLoss(T_0=1, T_1=2)),
    ('σD_T1_3',    SigmaDampLoss(T_0=1, T_1=3)),
    ('σD_T1_05',   SigmaDampLoss(T_0=1, T_1=0.5)),
]

hp_results = []

for config_name, loss_fn in configs:
    print(f"\n{'='*60}")
    print(f"  WB Finetune HP: {config_name}")
    print(f"{'='*60}")

    torch.manual_seed(0)
    model = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)
    model.fc = nn.Linear(2048, 1)
    model = model.to(device)

    optimizer = optim.Adam(model.parameters(), lr=1e-5, weight_decay=0.1)

    best_val_wga = -1
    best_state = None
    patience_counter = 0

    for epoch in range(50):
        train_finetune_one_epoch(model, train_loader, optimizer, loss_fn, device)
        _, val_wga, _ = evaluate_finetune(model, val_loader, device)

        if (epoch + 1) % 10 == 0:
            print(f"  Epoch {epoch+1}/50 | val_wga={val_wga:.3f}")

        if val_wga > best_val_wga:
            best_val_wga = val_wga
            best_state = copy.deepcopy(model.state_dict())
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= 15:
                print(f"  Early stopping at epoch {epoch+1}")
                break

    if best_state:
        model.load_state_dict(best_state)

    test_avg, test_wga, test_groups = evaluate_finetune(model, test_loader, device)
    print(f"  => val_wga={best_val_wga:.3f} | test_wga={test_wga:.3f} | test_avg={test_avg:.3f}")

    hp_results.append({
        'config': config_name,
        'val_wga': best_val_wga,
        'test_wga': test_wga,
        'test_avg': test_avg,
    })

df_hp_wb_ft = pd.DataFrame(hp_results)
df_hp_wb_ft.to_csv(f'{DRIVE_ROOT}/csv/results_waterbirds_hp_finetune.csv', index=False)
print(df_hp_wb_ft.to_string(index=False, float_format='{:.3f}'.format))


  WB Finetune HP: SD_lam005
Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /root/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth


100%|██████████| 97.8M/97.8M [00:01<00:00, 88.6MB/s]


  Epoch 10/50 | val_wga=0.113
  Epoch 20/50 | val_wga=0.444
  Epoch 30/50 | val_wga=0.496
  Epoch 40/50 | val_wga=0.451
  Epoch 50/50 | val_wga=0.556
  => val_wga=0.556 | test_wga=0.668 | test_avg=0.872

  WB Finetune HP: SD_lam01
  Epoch 10/50 | val_wga=0.135
  Epoch 20/50 | val_wga=0.368
  Epoch 30/50 | val_wga=0.466
  Epoch 40/50 | val_wga=0.436
  Epoch 50/50 | val_wga=0.526
  => val_wga=0.526 | test_wga=0.621 | test_avg=0.871

  WB Finetune HP: SD_lam05
  Epoch 10/50 | val_wga=0.233
  Early stopping at epoch 16
  => val_wga=0.353 | test_wga=0.352 | test_avg=0.608

  WB Finetune HP: ML_lam01
  Epoch 10/50 | val_wga=0.233
  Epoch 20/50 | val_wga=0.571
  Epoch 30/50 | val_wga=0.586
  Early stopping at epoch 38
  => val_wga=0.624 | test_wga=0.685 | test_avg=0.883

  WB Finetune HP: ML_lam05
  Epoch 10/50 | val_wga=0.227
  Epoch 20/50 | val_wga=0.440
  Epoch 30/50 | val_wga=0.530
  Epoch 40/50 | val_wga=0.612
  Epoch 50/50 | val_wga=0.530
  => val_wga=0.678 | test_wga=0.683 | test_avg=0

In [ ]:
#  Waterbirds Finetuning TESTING  # ACTUAL
# Update these after seeing HP results from Cell 1
# For now using same as SpurBreast tuned values
from torch.utils.data import Subset

methods = {
    'ERM':      LogLoss(),
    'SD':       SpectralDecoupling(lam=0.05),                      # Best SD WGA (0.668)
    'Marg-Log': MargLogLoss(lam=4.0, gamma_0=0.0, gamma_1=1.0),    # Best ML WGA (0.734)
    'σ-damp':   SigmaDampLoss(T_0=1.0, T_1=3.0),                   # Best σD WGA (0.621)
}

DATA_DIR = '/content/drive/MyDrive'
dataset, splits = load_waterbirds_wilds(DATA_DIR)

train_loader = DataLoader(splits['train'], batch_size=64, shuffle=True, num_workers=4)
val_loader = DataLoader(splits['val'], batch_size=64, shuffle=False, num_workers=4)
test_loader = DataLoader(splits['test'], batch_size=64, shuffle=False, num_workers=4)

# Get train metadata for subsampling
train_data = splits['train']
train_labels = np.array([train_data[i][1] if not torch.is_tensor(train_data[i][1]) else train_data[i][1].item() for i in range(len(train_data))])
train_groups = np.array([2 * train_labels[i] + (train_data[i][2][0].item() if torch.is_tensor(train_data[i][2][0]) else train_data[i][2][0]) for i in range(len(train_data))])

val_loader = DataLoader(splits['val'], batch_size=64, shuffle=False, num_workers=4)
test_loader = DataLoader(splits['test'], batch_size=64, shuffle=False, num_workers=4)

conditions = {
    'random_005': ('random', 0.05),
    'random_010': ('random', 0.10),
    'random_025': ('random', 0.25),
    'random_050': ('random', 0.50),
    'full':       ('random', 1.0),
    'gb_005':     ('balanced', 0.05),
    'gb_010':     ('balanced', 0.10),
    'gb_025':     ('balanced', 0.25),
    'gb_050':     ('balanced', 0.50),
    'gb_full':    ('balanced', 1.0),
}

all_results = []

for cond_name, (sampling, ratio) in conditions.items():
    rng = np.random.RandomState(42)
    n = len(train_labels)

    if sampling == 'random':
        n_keep = max(int(n * ratio), 2)
        indices = rng.choice(n, size=n_keep, replace=False)
    else:
        unique_g = np.unique(train_groups)
        min_size = min(np.sum(train_groups == g) for g in unique_g)
        target_per_group = max(int(min_size * ratio), 2)
        indices = []
        for g in unique_g:
            g_idx = np.where(train_groups == g)[0]
            chosen = rng.choice(g_idx, size=min(target_per_group, len(g_idx)), replace=False)
            indices.extend(chosen)
        indices = np.array(indices)
        rng.shuffle(indices)

    train_subset = Subset(train_data, indices)
    n_train = len(indices)
    d_over_n = 2048 / n_train
    shortcut_ratio = np.mean(np.isin(train_groups[indices], [0, 3]))

    train_loader_sub = DataLoader(train_subset, batch_size=64, shuffle=True, num_workers=4)

    for method_name, loss_fn in methods.items():
        print(f"\n{'='*60}")
        print(f"  {cond_name} | {method_name} | n={n_train} | d/n={d_over_n:.3f} | ρ={shortcut_ratio:.3f}")
        print(f"{'='*60}")

        torch.manual_seed(0)
        model = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)
        model.fc = nn.Linear(2048, 1)
        model = model.to(device)

        optimizer = optim.Adam(model.parameters(), lr=1e-5, weight_decay=0.1)

        best_val_wga = -1
        best_state = None
        patience_counter = 0

        for epoch in range(50):
            model.train()
            for x, y, metadata in train_loader_sub:
                x, y = x.to(device), y.to(device).float()
                optimizer.zero_grad()
                logits = model(x).squeeze(-1)
                loss = loss_fn(logits, y)
                loss.backward()
                optimizer.step()

            _, val_wga, _ = evaluate_finetune(model, val_loader, device)

            if (epoch + 1) % 10 == 0:
                print(f"  Epoch {epoch+1}/50 | val_wga={val_wga:.3f}")

            if val_wga > best_val_wga:
                best_val_wga = val_wga
                best_state = copy.deepcopy(model.state_dict())
                patience_counter = 0
            else:
                patience_counter += 1
                if patience_counter >= 15:
                    print(f"  Early stopping at epoch {epoch+1}")
                    break

        if best_state:
            model.load_state_dict(best_state)

        test_avg, test_wga, _ = evaluate_finetune(model, test_loader, device)
        print(f"  => test_wga={test_wga:.3f} | test_avg={test_avg:.3f}")

        all_results.append({
            'condition': cond_name,
            'sampling': sampling,
            'method': method_name,
            'n_train': n_train,
            'd_over_n': d_over_n,
            'shortcut_ratio': shortcut_ratio,
            'val_wga': best_val_wga,
            'test_wga': test_wga,
            'test_avg': test_avg,
        })

DRIVE_ROOT = '/content/drive/MyDrive'
df_wb_ft = pd.DataFrame(all_results)
df_wb_ft.to_csv(f'{DRIVE_ROOT}/csv/results_waterbirds_finetuning.csv', index=False)
print(df_wb_ft.to_string(index=False, float_format='{:.3f}'.format))

In [ ]:
# NEED DOWNLOADED SPURBREAST in drive/MyDrive FROM GITHUB:
# https://github.com/utkuozbulak/spurbreast
# we used field strength folder

In [ ]:
#  SpurBreast Hyperparameter Tuning (Linear Probe)  # ACTUAL

# Extract embeddings (need the old extraction path)
sb_splits, sb_group_names_train, sb_group_names_test = extract_spurbreast_features(
    fs_dir='/content/drive/MyDrive/field_strength/field_strength',
    vf_dir='/content/drive/MyDrive/vertical_flip',
    baseline_dir='/content/drive/MyDrive/baseline_low',
    rho2=0.9,
    flip_seed=42,
)

# Bias field strength on embeddings
# y_tr is {-1,1}, convert to {0,1} for comparison with z1
y_01 = ((y_tr + 1) / 2).astype(int)

# Recover z1 from groups: groups = 2*agree1 + agree2
# agree1=1 means z1==y, so groups 2 and 3 have agree1=1
# We need actual z1 values, not just agreement
# From the group structure: if agree1=1 then z1=y, if agree1=0 then z1=1-y
agree1 = g_tr // 2  # 0 or 1
z1_tr = np.where(agree1 == 1, y_01, 1 - y_01)

# Now bias
rng = np.random.RandomState(42)
agree_idx = np.where(y_01 == z1_tr)[0]
disagree_idx = np.where(y_01 != z1_tr)[0]

rho1 = 0.9
n_disagree_needed = int(len(agree_idx) * (1 - rho1) / rho1)

if n_disagree_needed <= len(disagree_idx):
    keep_agree = agree_idx
    keep_disagree = rng.choice(disagree_idx, size=n_disagree_needed, replace=False)
else:
    n_agree_needed = int(len(disagree_idx) * rho1 / (1 - rho1))
    keep_agree = rng.choice(agree_idx, size=n_agree_needed, replace=False)
    keep_disagree = disagree_idx

keep_idx = np.concatenate([keep_agree, keep_disagree])
rng.shuffle(keep_idx)

print(f"  Field strength biasing: {len(y_tr)} -> {len(keep_idx)} images")
print(f"  z1 correlation: {np.mean(y_01 == z1_tr):.3f} -> {np.mean(y_01[keep_idx] == z1_tr[keep_idx]):.3f}")

X_tr = X_tr[keep_idx]
y_tr = y_tr[keep_idx]
g_tr = g_tr[keep_idx]
X_te = sb_splits['test']['X']
y_te = sb_splits['test']['y']
g_te = sb_splits['test']['groups']

y_tr = ((y_tr + 1) / 2).astype(np.float32)
y_te = ((y_te + 1) / 2).astype(np.float32)

# Actually simpler: just run on full data without biasing for HP tuning
# The point is to find which λ works best, not to match the exact training distribution

d = X_tr.shape[1]

configs = [
    ('SD_lam005',  SpectralDecoupling(lam=0.05)),
    ('SD_lam01',   SpectralDecoupling(lam=0.1)),
    ('SD_lam05',   SpectralDecoupling(lam=0.5)),
    ('ML_lam001_g2',  MargLogLoss(lam=0.01, gamma_0=0, gamma_1=2)),
    ('ML_lam005_g2',  MargLogLoss(lam=0.05, gamma_0=0, gamma_1=2)),
    ('ML_lam01_g2',   MargLogLoss(lam=0.1,  gamma_0=0, gamma_1=2)),
    ('ML_lam05_g2',   MargLogLoss(lam=0.5,  gamma_0=0, gamma_1=2)),
    ('ML_lam1_g2',    MargLogLoss(lam=1.0,  gamma_0=0, gamma_1=2)),
    ('ML_g1_05',   MargLogLoss(lam=0.1, gamma_0=0, gamma_1=0.5)),
    ('ML_g1_1',    MargLogLoss(lam=0.1, gamma_0=0, gamma_1=1)),
    ('ML_g1_2',    MargLogLoss(lam=0.1, gamma_0=0, gamma_1=2)),
    ('σD_T1_05',  SigmaDampLoss(T_0=1, T_1=0.5)),
    ('σD_T1_1',   SigmaDampLoss(T_0=1, T_1=1)),
    ('σD_T1_2',   SigmaDampLoss(T_0=1, T_1=2)),
    ('σD_T1_3',   SigmaDampLoss(T_0=1, T_1=3)),
    ('σD_T0_05',  SigmaDampLoss(T_0=0.5, T_1=2)),
    ('σD_T0_1',   SigmaDampLoss(T_0=1,   T_1=2)),
    ('σD_T0_2',   SigmaDampLoss(T_0=2,   T_1=2)),
]

hp_results = []

for config_name, loss_factory in configs:
    print(f"\n{'='*60}")
    print(f"  SpurBreast HP: {config_name}")
    print(f"{'='*60}")

    torch.manual_seed(0)
    model = nn.Linear(d, 1, bias=False).to(device)
    optimizer = optim.Adam(model.parameters(), lr=0.01)

    Xt = torch.tensor(X_tr, dtype=torch.float32).to(device)
    yt = torch.tensor(y_tr, dtype=torch.float32).to(device)

    loss_fn = loss_factory
    for epoch in range(3000):
        optimizer.zero_grad()
        logits = model(Xt).squeeze()
        loss = loss_fn(logits, yt)
        loss.backward()
        optimizer.step()

    model.eval()
    with torch.no_grad():
        test_logits = model(torch.tensor(X_te, dtype=torch.float32).to(device)).squeeze().cpu().numpy()
        test_preds = (test_logits > 0).astype(float)

    metrics = compute_group_metrics(test_preds, y_te, g_te, sb_group_names_test)
    print(f"  => avg_acc={metrics['avg_acc']:.3f} | wga={metrics['wga']:.3f}")

    hp_results.append({
        'config': config_name,
        'avg_acc': metrics['avg_acc'],
        'wga': metrics['wga'],
    })

df_hp_sb = pd.DataFrame(hp_results)
save_csv(df_hp_sb, 'results_spurbreast_hyperparam_linear.csv')
print(df_hp_sb.to_string(index=False, float_format='{:.3f}'.format))


  Processing train (field_strength biased + injected flip, rho2=0.9)...
      9563 images (pos=4782, neg=4781)
        640/9563
        1280/9563
        1920/9563
        2560/9563
        3200/9563
        3840/9563
        4480/9563
        5120/9563
        5760/9563
        6400/9563
        7040/9563
        7680/9563
        8320/9563
        8960/9563
    X: (9563, 2048)
    z1 (field strength) correlation: 0.513
    z2 (vertical flip) correlation:  0.903
    Groups: {np.int64(0): np.int64(441), np.int64(1): np.int64(4218), np.int64(2): np.int64(483), np.int64(3): np.int64(4421)}

  Processing test (baseline — unbiased)...
    Baseline:
      6788 images (pos=3394, neg=3394)
        640/6788
        1280/6788
        1920/6788
        2560/6788
        3200/6788
        3840/6788
        4480/6788
        5120/6788
        5760/6788
        6400/6788
    test: X=(6788, 2048), groups={np.int64(0): np.int64(3394), np.int64(1): np.int64(3394)}
  Field strength biasing: 5448 -> 54

In [ ]:
def run_hyperparam_sweep(train_dataset, val_dataset, test_dataset, device='cuda'):  # ACTUAL HYPERPARAMETER FINETUNING SPURBREAST
    val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False, num_workers=4, pin_memory=True)
    test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False, num_workers=4, pin_memory=True)

    d = 2048

    configs = [
        ('SD_lam005',  SpectralDecoupling(lam=0.05)),
        ('SD_lam01',   SpectralDecoupling(lam=0.1)),
        ('SD_lam05',   SpectralDecoupling(lam=0.5)),
        ('ML_lam001_g2',  MargLogLoss(lam=0.01, gamma_0=0, gamma_1=2)),
        ('ML_lam005_g2',  MargLogLoss(lam=0.05, gamma_0=0, gamma_1=2)),
        ('ML_lam01_g2',   MargLogLoss(lam=0.1,  gamma_0=0, gamma_1=2)),
        ('ML_lam05_g2',   MargLogLoss(lam=0.5,  gamma_0=0, gamma_1=2)),
        ('ML_lam1_g2',    MargLogLoss(lam=1.0,  gamma_0=0, gamma_1=2)),
        ('ML_g1_05',   MargLogLoss(lam=0.1, gamma_0=0, gamma_1=0.5)),
        ('ML_g1_1',    MargLogLoss(lam=0.1, gamma_0=0, gamma_1=1)),
        ('ML_g1_2',    MargLogLoss(lam=0.1, gamma_0=0, gamma_1=2)),
        ('σD_T1_05',  SigmaDampLoss(T_0=1, T_1=0.5)),
        ('σD_T1_1',   SigmaDampLoss(T_0=1, T_1=1)),
        ('σD_T1_2',   SigmaDampLoss(T_0=1, T_1=2)),
        ('σD_T1_3',   SigmaDampLoss(T_0=1, T_1=3)),
        ('σD_T0_05',  SigmaDampLoss(T_0=0.5, T_1=2)),
        ('σD_T0_1',   SigmaDampLoss(T_0=1,   T_1=2)),
        ('σD_T0_2',   SigmaDampLoss(T_0=2,   T_1=2)),
    ]

    all_results = []

    for config_name, loss_fn in configs:
        print(f"\n{'='*60}")
        print(f"  Hyperparam sweep: {config_name}")
        print(f"{'='*60}")

        torch.manual_seed(0)
        model = create_resnet_model(device)

        train_loader = DataLoader(train_dataset, batch_size=64,
                                  shuffle=True, num_workers=4, pin_memory=True)

        model, val_wga = train_finetune_spurbreast(
            model, train_loader, val_loader, loss_fn,
            lr=1e-5, weight_decay=0.1, epochs=50, patience=20, device=device
        )

        test_avg, test_wga, test_groups = evaluate_model(model, test_loader, device)
        print(f"  => val_wga={val_wga:.3f} | test_wga={test_wga:.3f} | test_avg={test_avg:.3f}")

        all_results.append({
            'config': config_name,
            'val_wga': val_wga,
            'test_wga': test_wga,
            'test_avg': test_avg,
        })

    return pd.DataFrame(all_results)

train_dataset, val_dataset, test_dataset = prepare_spurbreast_datasets(
    fs_dir='/content/local_data/field_strength',
    baseline_dir='/content/local_data/baseline_low',
    clinical_xlsx_path='/content/local_data/Clinical_and_Other_Features.xlsx',
    rho2=0.9,
    flip_seed=42,
)
df_hp = run_hyperparam_sweep(train_dataset, val_dataset, test_dataset, device='cuda')
DRIVE_ROOT = '/content/drive/MyDrive/shortcut_experiments'
os.makedirs(f'{DRIVE_ROOT}/csv', exist_ok=True)
df_hp.to_csv(f'{DRIVE_ROOT}/csv/results_spurbreast_hyperparam.csv', index=False)
print(df_hp.to_string(index=False, float_format='{:.3f}'.format))

DUKE metadata: 922 patients, 1.5T=468, 3T=454
  Field strength biasing: 9563 -> 5448 images
  z1 correlation: 0.513 -> 0.900
Training: 5448 images
  z1 (field strength) correlation: 0.900
  z2 (vertical flip) correlation:  0.902
  Groups: {np.int64(0): np.int64(50), np.int64(1): np.int64(494), np.int64(2): np.int64(483), np.int64(3): np.int64(4421)}
Test: 6788 images

  Hyperparam sweep: SD_lam005
  Epoch 5/50 | train_loss=0.4684 | val_avg=0.903 | val_wga=0.159
  Epoch 10/50 | train_loss=0.4010 | val_avg=0.912 | val_wga=0.252
  Epoch 15/50 | train_loss=0.3714 | val_avg=0.928 | val_wga=0.346
  Epoch 20/50 | train_loss=0.3469 | val_avg=0.944 | val_wga=0.505
  Epoch 25/50 | train_loss=0.3319 | val_avg=0.957 | val_wga=0.645
  Epoch 30/50 | train_loss=0.3244 | val_avg=0.965 | val_wga=0.701
  Epoch 35/50 | train_loss=0.3207 | val_avg=0.961 | val_wga=0.673
  Epoch 40/50 | train_loss=0.3199 | val_avg=0.961 | val_wga=0.692
  Epoch 45/50 | train_loss=0.3203 | val_avg=0.965 | val_wga=0.729
  Epoc

In [ ]:
"""
SPURBREAST 2-SHORTCUT EXPERIMENTS
Tests MARG-CTRL vs ERM on a dataset with two shortcuts:
  z1: Field Strength (1.5T vs 3T) - Real
  z2: Vertical Flip - Injected
"""

def get_file_paths(folder, label):
    paths = []
    for ext in ['*.png', '*.jpg', '*.jpeg']:
        paths.extend(glob.glob(os.path.join(folder, ext)))
    return [(p, label) for p in sorted(paths)]

def load_duke_field_strength(clinical_xlsx_path):
    if not os.path.exists(clinical_xlsx_path):
        print(f"WARNING: Duke clinical file not found at {clinical_xlsx_path}. Assuming all 1.5T.")
        return {}
    duke_df = pd.read_excel(clinical_xlsx_path, header=1, skiprows=[2])
    fs_map = {}
    for _, row in duke_df.iterrows():
        pat_id = str(row['Patient ID'])
        pat_num = int(pat_id.split('_')[-1])
        fs_code = int(row['Field Strength (Tesla)'])
        fs_map[pat_num] = 1 if fs_code >= 2 else 0
    return fs_map

def extract_spurbreast_features(fs_dir, baseline_dir, clinical_xlsx_path, rho2=0.7, flip_seed=42, device='cuda'):
    fs_map = load_duke_field_strength(clinical_xlsx_path)

    def get_fs_label(filename):
        try:
            pat_num = int(os.path.basename(filename).split('-')[1])
            return fs_map.get(pat_num, 0)
        except:
            return 0

    resnet = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)
    resnet.fc = nn.Identity()
    resnet = resnet.to(device).eval()

    def extract_from_folder(split_dir):
        pos_paths = get_file_paths(os.path.join(split_dir, '1'), 1)
        neg_paths = get_file_paths(os.path.join(split_dir, '0'), 0)
        all_paths = pos_paths + neg_paths

        if len(all_paths) == 0:
            return None, None

        all_features, batch, labels = [], [], []
        for i, (path, label) in enumerate(all_paths):
            im = Image.open(path).convert('RGB').resize((224, 224))
            im_tensor = torch.tensor(np.array(im), dtype=torch.float32).permute(2, 0, 1) / 255.0
            batch.append(im_tensor)
            labels.append(label)

            if len(batch) == 64 or i == len(all_paths) - 1:
                with torch.no_grad():
                    feats = resnet(torch.stack(batch).to(device)).cpu().numpy()
                all_features.append(feats)
                batch = []

        return np.concatenate(all_features).astype(np.float32), np.array(labels, dtype=np.float32)

    def extract_with_flip(split_dir, rho2, seed):
        rng = np.random.RandomState(seed)
        pos_paths = get_file_paths(os.path.join(split_dir, '1'), 1)
        neg_paths = get_file_paths(os.path.join(split_dir, '0'), 0)
        all_paths = pos_paths + neg_paths

        if len(all_paths) == 0:
            raise ValueError(f"No images found in {split_dir}")

        all_features, batch, y_list, z1_list, z2_list = [], [], [], [], []

        for i, (path, label) in enumerate(all_paths):
            fname = os.path.basename(path)
            im = Image.open(path).convert('RGB').resize((224, 224))

            z1_val = get_fs_label(fname)
            z2_val = label if rng.random() < rho2 else 1 - label

            if z2_val == 1:
                im = im.transpose(Image.FLIP_TOP_BOTTOM)

            im_tensor = torch.tensor(np.array(im), dtype=torch.float32).permute(2, 0, 1) / 255.0
            batch.append(im_tensor)
            y_list.append(label)
            z1_list.append(z1_val)
            z2_list.append(z2_val)

            if len(batch) == 64 or i == len(all_paths) - 1:
                with torch.no_grad():
                    feats = resnet(torch.stack(batch).to(device)).cpu().numpy()
                all_features.append(feats)
                batch = []

        return np.concatenate(all_features).astype(np.float32), np.array(y_list, dtype=np.float32), np.array(z1_list), np.array(z2_list)

    print("Extracting Training Set (with Flips & Field Strength)...")
    X_tr, y_tr, z1_tr, z2_tr = extract_with_flip(os.path.join(fs_dir, 'training'), rho2, flip_seed)

    agree1 = (y_tr == z1_tr).astype(int)
    agree2 = (y_tr == z2_tr).astype(int)
    groups_tr = 2 * agree1 + agree2

    print("Extracting Test Set (Baseline Unbiased)...")
    X_te, y_te = extract_from_folder(os.path.join(baseline_dir, 'test'))
    # On the unbiased test set, WGA is just min(benign_acc, malignant_acc)
    groups_te = y_te.astype(int)

    # Make a quick 80/20 train/val split for early stopping
    np.random.seed(42)
    indices = np.random.permutation(len(y_tr))
    split = int(len(y_tr) * 0.8)
    tr_idx, val_idx = indices[:split], indices[split:]

    embeddings = {
        'train': {'X': X_tr[tr_idx], 'y': y_tr[tr_idx], 'groups': groups_tr[tr_idx]},
        'val': {'X': X_tr[val_idx], 'y': y_tr[val_idx], 'groups': groups_tr[val_idx]},
        'test': {'X': X_te, 'y': y_te, 'groups': groups_te}
    }
    return embeddings


# LINEAR PROBE & SAMPLING ALGORITHMS

def run_linear_probe_fixed(X_train, y_train, groups_train,
                           X_val, y_val, groups_val,
                           X_test, y_test, groups_test,
                           loss_fn, lr=0.01, epochs=5000,
                           device='cuda', use_bias=True):
    d = X_train.shape[1]

    X_tr = torch.tensor(X_train, dtype=torch.float32, device=device)
    y_tr = torch.tensor(y_train, dtype=torch.long, device=device)
    g_tr = torch.tensor(groups_train, dtype=torch.long, device=device)
    X_v = torch.tensor(X_val, dtype=torch.float32, device=device)
    y_v = torch.tensor(y_val, dtype=torch.long, device=device)
    g_v = torch.tensor(groups_val, dtype=torch.long, device=device)
    X_te = torch.tensor(X_test, dtype=torch.float32, device=device)
    y_te = torch.tensor(y_test, dtype=torch.long, device=device)
    g_te = torch.tensor(groups_test, dtype=torch.long, device=device)

    model = nn.Linear(d, 1, bias=use_bias).to(device)
    optimizer = optim.Adam(model.parameters(), lr=lr)

    best_val_wga = -1
    best_state = None
    patience_counter = 0
    patience = 500

    for epoch in range(epochs):
        model.train()
        optimizer.zero_grad()
        logits = model(X_tr).squeeze(-1)
        loss = loss_fn(logits, y_tr)
        loss.backward()
        optimizer.step()

        if (epoch + 1) % 50 == 0:
            model.eval()
            with torch.no_grad():
                val_logits = model(X_v).squeeze(-1)
                val_preds = (val_logits > 0).long()
                val_correct = (val_preds == y_v).float()

                val_group_accs = [val_correct[g_v == g].mean().item() for g in g_v.unique() if (g_v == g).sum() > 0]
                val_wga = min(val_group_accs) if val_group_accs else 0.0

                if val_wga > best_val_wga:
                    best_val_wga = val_wga
                    best_state = copy.deepcopy(model.state_dict())
                    patience_counter = 0
                else:
                    patience_counter += 50

                if patience_counter >= patience: break

    if best_state: model.load_state_dict(best_state)

    model.eval()
    with torch.no_grad():
        test_logits = model(X_te).squeeze(-1)
        test_preds = (test_logits > 0).long()
        test_correct = (test_preds == y_te).float()
        test_avg = test_correct.mean().item()
        test_group_accs = {g.item(): test_correct[g_te == g].mean().item() for g in g_te.unique()}
        test_wga = min(test_group_accs.values())

    return {'test_wga': test_wga, 'val_wga': best_val_wga, 'test_avg': test_avg}


def get_balanced_subset(X, y, groups, k_per_group):
    unique_groups = np.unique(groups)
    indices = []
    for g in unique_groups:
        g_idx = np.where(groups == g)[0]
        if len(g_idx) >= k_per_group:
            chosen = np.random.choice(g_idx, size=k_per_group, replace=False)
        else:
            chosen = np.random.choice(g_idx, size=k_per_group, replace=True)
        indices.extend(chosen)
    indices = np.array(indices)
    shuffle_idx = np.random.permutation(len(indices))
    return X[indices[shuffle_idx]], y[indices[shuffle_idx]], groups[indices[shuffle_idx]]

def gb_downsample(X, y, groups):
    min_size = min(np.sum(groups == g) for g in np.unique(groups))
    return get_balanced_subset(X, y, groups, min_size)

def gb_oversample(X, y, groups):
    max_size = max(np.sum(groups == g) for g in np.unique(groups))
    return get_balanced_subset(X, y, groups, max_size)

# EXPERIMENT SUITES

def get_loss_methods():
    return {
        'ERM': LogLoss(),
        'SD': SpectralDecoupling(lam=0.05),
        'Marg-Log': MargLogLoss(lam=0.5, gamma_0=0, gamma_1=2),
        'σ-damp': SigmaDampLoss(T_0=0.5, T_1=2),
    }

def run_fixed_subsample(embeddings, device='cuda'):
    scaler = StandardScaler()
    X_tr = scaler.fit_transform(embeddings['train']['X']).astype(np.float32)
    X_val = scaler.transform(embeddings['val']['X']).astype(np.float32)
    X_te = scaler.transform(embeddings['test']['X']).astype(np.float32)

    y_tr, g_tr = embeddings['train']['y'], embeddings['train']['groups']

    conditions = ['Full', 'GB-Down', 'GB-Over', 'Random (r=0.25)']
    methods = get_loss_methods()
    all_results = []

    for cond in conditions:
        if cond == 'Full': X_c, y_c, g_c = X_tr, y_tr, g_tr
        elif cond == 'GB-Down': X_c, y_c, g_c = gb_downsample(X_tr, y_tr, g_tr)
        elif cond == 'GB-Over': X_c, y_c, g_c = gb_oversample(X_tr, y_tr, g_tr)
        elif cond == 'Random (r=0.25)':
            idx = np.random.choice(len(y_tr), size=len(y_tr)//4, replace=False)
            X_c, y_c, g_c = X_tr[idx], y_tr[idx], g_tr[idx]

        print(f"\n{'='*60}\n  {cond}: n={len(y_c)}, d/n={X_c.shape[1]/len(y_c):.3f}\n{'='*60}")

        for name, loss_fn in methods.items():
            seed_wgas = []
            for seed in range(3):
                np.random.seed(seed); torch.manual_seed(seed)
                res = run_linear_probe_fixed(X_c, y_c, g_c, X_val, embeddings['val']['y'], embeddings['val']['groups'],
                                             X_te, embeddings['test']['y'], embeddings['test']['groups'],
                                             loss_fn, device=device)
                seed_wgas.append(res['test_wga'])
                all_results.append({'condition': cond, 'method': name, 'seed': seed, 'test_wga': res['test_wga'], 'd_over_n': X_c.shape[1]/len(y_c)})
            print(f"  {name:>10s} | WGA = {np.mean(seed_wgas):.3f} ± {np.std(seed_wgas):.3f}")
    return pd.DataFrame(all_results)


def run_dn_balanced_sweep(embeddings, device='cuda'):
    scaler = StandardScaler()
    X_tr = scaler.fit_transform(embeddings['train']['X']).astype(np.float32)
    X_val = scaler.transform(embeddings['val']['X']).astype(np.float32)
    X_te = scaler.transform(embeddings['test']['X']).astype(np.float32)

    y_tr, g_tr = embeddings['train']['y'], embeddings['train']['groups']

    _, counts = np.unique(g_tr, return_counts=True)
    k_targets = {
        'Extreme Small': 25,
        'GB-Down Exact': int(np.min(counts)),
        'Medium Balanced': 250,
        'Full Data Eq': len(y_tr) // 4,
        'GB-Over Exact': int(np.max(counts))
    }

    methods = get_loss_methods()
    all_results = []

    for label, k in k_targets.items():
        n_total = k * 4
        print(f"\n{'='*70}\n  Target: {label} | N={n_total} (K={k}/grp) | d/n = {X_tr.shape[1]/n_total:.3f}\n{'='*70}")

        for samp_type in ['Random', 'Balanced']:
            print(f"\n  --- Sampling: {samp_type} ---")
            for name, loss_fn in methods.items():
                seed_wgas = []
                for seed in range(3):
                    np.random.seed(seed); torch.manual_seed(seed)
                    if samp_type == 'Balanced':
                        X_c, y_c, g_c = get_balanced_subset(X_tr, y_tr, g_tr, k)
                    else:
                        idx = np.random.choice(len(y_tr), size=n_total, replace=(n_total>len(y_tr)))
                        X_c, y_c, g_c = X_tr[idx], y_tr[idx], g_tr[idx]

                    res = run_linear_probe_fixed(X_c, y_c, g_c, X_val, embeddings['val']['y'], embeddings['val']['groups'],
                                                 X_te, embeddings['test']['y'], embeddings['test']['groups'],
                                                 loss_fn, device=device)
                    seed_wgas.append(res['test_wga'])
                    all_results.append({'target': label, 'sampling': samp_type, 'method': name, 'seed': seed,
                                        'test_wga': res['test_wga'], 'd_over_n': X_tr.shape[1]/n_total})
                print(f"    {name:>10s} | WGA = {np.mean(seed_wgas):.3f} ± {np.std(seed_wgas):.3f}")
    return pd.DataFrame(all_results)

# MAIN

if __name__ == '__main__':
    import argparse
    parser = argparse.ArgumentParser()
    parser.add_argument('--mode', choices=['fix_subsample', 'dn_balanced'], default='dn_balanced')
    parser.add_argument('--fs_dir', default='/content/local_data/field_strength')
    parser.add_argument('--baseline_dir', default='/content/local_data/baseline_low')
    parser.add_argument('--clinical_xlsx', default='/content/local_data/Clinical_and_Other_Features.xlsx')

    args, _ = parser.parse_known_args()
    device = 'cuda' if torch.cuda.is_available() else 'cpu'

    # MANUAL OVERRIDE FOR COLAB
    args.mode = 'fix_subsample'

    print("=" * 70)
    print("  EXTRACTING SPURBREAST FEATURES")
    print("=" * 70)
    embeddings = extract_spurbreast_features(
        fs_dir=args.fs_dir,
        baseline_dir=args.baseline_dir,
        clinical_xlsx_path=args.clinical_xlsx,
        rho2=0.9,
        device=device
    )

    if args.mode == 'fix_subsample':
        df = run_fixed_subsample(embeddings, device)
        df.to_csv('spurbreast_subsample.csv', index=False) # LINEAR PROBE TESTING SPURBREAST RANDOM SAMPLING
    elif args.mode == 'dn_balanced':
        df = run_dn_balanced_sweep(embeddings, device)
        df.to_csv('spurbreast_dn_sweep.csv', index=False) # LINEAR PROBE TESTING SPURBREAST GROUP BALANCED

In [11]:
## DATASETS:
from torch.utils.data.dataset import Dataset
from torchvision import transforms, models

def get_file_paths(folder, label): # used in extract spurbreast function
    paths = []
    for ext in ['*.png', '*.jpg', '*.jpeg']:
        paths.extend(glob.glob(os.path.join(folder, ext)))
    return [(p, label) for p in sorted(paths)]


# ─── DUKE field strength metadata ───
def load_duke_field_strength(clinical_xlsx_path): # load DUKE clinical excel and return patient_num for field strength_binary mapping so i know what samples use what field strnegth
    duke_df = pd.read_excel(clinical_xlsx_path, header=1, skiprows=[2])
    fs_map = {}
    for _, row in duke_df.iterrows():
        pat_id = str(row['Patient ID'])
        pat_num = int(pat_id.split('_')[-1])
        fs_code = int(row['Field Strength (Tesla)'])
        fs_map[pat_num] = 1 if fs_code >= 2 else 0  # 0=1.5T, 1=3T
    print(f"DUKE metadata: {len(fs_map)} patients, 1.5T={sum(1 for v in fs_map.values() if v==0)}, 3T={sum(1 for v in fs_map.values() if v==1)}") # i know how many samples use 1.5T and 3T?
    return fs_map

def get_patient_num(filename):
    return int(os.path.basename(filename).split('-')[1])

def get_fs_label(filename):
    return fs_map.get(get_patient_num(filename), 0) # returns 0 for 1.5T and 1 for 3T field strength

# Load this ONCE at the top level
fs_map = load_duke_field_strength('/content/drive/MyDrive/Clinical_and_Other_Features.xlsx') # contains info required for load duke field strength func


# ─── SpurBreast feature extraction with two shortcuts ───

def extract_spurbreast_features(fs_dir='/content/data/spurbreast/field_strength',
                                 vf_dir='/content/data/spurbreast/vertical_flip',
                                 baseline_dir='/content/data/spurbreast/baseline_large',
                                 rho2=0.7, flip_seed=42):

    resnet = models.resnet50(pretrained=True)
    resnet.fc = nn.Identity() # get rid of last layer for embeddings
    resnet.eval().to(device)

    def extract_from_folder(split_dir):
        pos_paths = get_file_paths(os.path.join(split_dir, '1'), 1) # gets from file '1' containing tumor samples
        neg_paths = get_file_paths(os.path.join(split_dir, '0'), 0)
        all_paths = pos_paths + neg_paths
        print(f'      {len(all_paths)} images (pos={len(pos_paths)}, neg={len(neg_paths)})')

        if len(all_paths) == 0:
            return None, None, {}

        all_features, batch = [], []
        image_paths = [p for p, _ in all_paths]
        labels = np.array([l for _, l in all_paths], dtype=np.float32)

        for i, path in enumerate(image_paths):
            im = Image.open(path).convert('RGB')
            im = im.resize((224, 224))
            im_tensor = torch.tensor(np.array(im), dtype=torch.float32).permute(2, 0, 1) / 255.0
            batch.append(im_tensor)

            if len(batch) == 64 or i == len(image_paths) - 1:
                with torch.no_grad():
                    feats = resnet(torch.stack(batch).to(device)).cpu().numpy() # get embeddings
                all_features.append(feats)
                batch = []
                if (i + 1) % (64 * 10) < 64:
                    print(f'        {i+1}/{len(image_paths)}')

        X = np.concatenate(all_features).astype(np.float32)
        label_lookup = {os.path.basename(p): int(l) for p, l in all_paths}
        return X, labels, label_lookup #  X contains embeddings where cols = embedding, rows = sample, size (N, 2048).
        # labels is size (N,)
        # label lookup (dont really need) is dict with key:value as string img filename: int label

    def extract_with_flip(split_dir, rho2, seed): # second shortcut for testing 2 shortcuts
        """
        Extract ResNet features, applying vertical flip to selected images
        to inject z2 shortcut. Returns X, y, z1, z2 arrays.
        """
        rng = np.random.RandomState(seed)

        pos_paths = get_file_paths(os.path.join(split_dir, '1'), 1)
        neg_paths = get_file_paths(os.path.join(split_dir, '0'), 0)
        all_paths = pos_paths + neg_paths
        print(f'      {len(all_paths)} images (pos={len(pos_paths)}, neg={len(neg_paths)})')

        if len(all_paths) == 0:
            return None, None, None, None

        all_features = []
        y_list, z1_list, z2_list = [], [], []
        batch = []

        for i, (path, label) in enumerate(all_paths):
            fname = os.path.basename(path)
            im = Image.open(path).convert('RGB').resize((224, 224))

            # z1: actual field strength from DUKE
            z1_val = get_fs_label(fname)

            # z2: inject flip with correlation rho2
            if rng.random() < rho2: # rho2 is the chance that malignant images will be flipped while benign stay normal so inducing shortcut for detecting malignant
                z2_val = label      # correlated with label
            else:
                z2_val = 1 - label  # anti-correlated

            # Apply the flip if z2=1 so correlated
            if z2_val == 1:
                im = im.transpose(Image.FLIP_TOP_BOTTOM)

            im_tensor = torch.tensor(np.array(im), dtype=torch.float32).permute(2, 0, 1) / 255.0
            batch.append(im_tensor)
            y_list.append(label)
            z1_list.append(z1_val)
            z2_list.append(z2_val)

            if len(batch) == 64 or i == len(all_paths) - 1:
                with torch.no_grad():
                    feats = resnet(torch.stack(batch).to(device)).cpu().numpy() # resnet embeddings
                all_features.append(feats)
                batch = []
                if (i + 1) % (64 * 10) < 64:
                    print(f'        {i+1}/{len(all_paths)}')

        X = np.concatenate(all_features).astype(np.float32)
        y = np.array(y_list, dtype=int)
        z1 = np.array(z1_list, dtype=int)
        z2 = np.array(z2_list, dtype=int)
        return X, y, z1, z2

    # ── Build training split with both shortcuts ──
    splits = {}

    print(f'\n  Processing train (field_strength biased + injected flip, rho2={rho2})...')
    fs_train_dir = os.path.join(fs_dir, 'training')
    X, y, z1, z2 = extract_with_flip(fs_train_dir, rho2=rho2, seed=flip_seed)

    y_signed = (y.astype(float) * 2 - 1).astype(np.float32)

    # 4 groups: (z1 agrees with y, z2 agrees with y)
    agree1 = (y == z1).astype(int)
    agree2 = (y == z2).astype(int)
    groups = 2 * agree1 + agree2

    print(f'    X: {X.shape}')
    print(f'    z1 (field strength) correlation: {np.mean(agree1):.3f}')
    print(f'    z2 (vertical flip) correlation:  {np.mean(agree2):.3f}')
    print(f'    Groups: {dict(zip(*np.unique(groups, return_counts=True)))}')

    splits['train'] = {'X': X, 'y': y_signed, 'groups': groups}

    # ── Test from baseline (unbiased, no flip) ──
    print(f'\n  Processing test (baseline — unbiased)...')
    print(f'    Baseline:')
    X_test, y_test_raw, _ = extract_from_folder(os.path.join(baseline_dir, 'test'))

    if X_test is not None:
        y_test = (y_test_raw * 2 - 1).astype(np.float32)
        groups_test = y_test_raw.astype(int)
        splits['test'] = {'X': X_test, 'y': y_test, 'groups': groups_test}
        print(f'    test: X={X_test.shape}, groups={dict(zip(*np.unique(groups_test, return_counts=True)))}')
    else:
        print('    WARNING: No baseline test found')
        print(f'    Field strength test:')
        X_fb, y_fb_raw, _ = extract_from_folder(os.path.join(fs_dir, 'test'))
        y_fb = (y_fb_raw * 2 - 1).astype(np.float32)
        splits['test'] = {'X': X_fb, 'y': y_fb, 'groups': y_fb_raw.astype(int)}

    # ── Group names ──
    group_names_train = {
        0: 'neither_agrees',      # z1≠y AND z2≠y
        1: 'z2_only_agrees',      # z1≠y AND z2=y
        2: 'z1_only_agrees',      # z1=y AND z2≠y
        3: 'both_agree',          # z1=y AND z2=y
    }
    group_names_test = {
        0: 'benign',
        1: 'malignant',
    }

    return splits, group_names_train, group_names_test

# PCA to reduce dimensions of resnet embeddings to try and concentrate the shortcut into single value in one of dimensions:

from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler


class LogLoss(nn.Module):
    """Standard ERM log-loss."""
    def forward(self, logits, labels):
        # labels in {0, 1}
        return nn.functional.binary_cross_entropy_with_logits(logits, labels.float())


class SpectralDecoupling(nn.Module):
    """SD loss: log-loss + λ * |f(x)|^2"""
    def __init__(self, lam=0.1):
        super().__init__()
        self.lam = lam

    def forward(self, logits, labels):
        bce = nn.functional.binary_cross_entropy_with_logits(logits, labels.float())
        penalty = self.lam * (logits ** 2).mean()
        return bce + penalty


class MargLogLoss(nn.Module):
    """Marg-Log loss with per-class targets: log-loss + λ * log(1 + |f - γ_y|^2)

    Per-class targets γ_0, γ_1 handle label imbalance.
    """
    def __init__(self, lam=0.1, gamma_0=0.0, gamma_1=1.0):
        super().__init__()
        self.lam = lam
        self.gamma_0 = gamma_0
        self.gamma_1 = gamma_1

    def forward(self, logits, labels):
        bce = nn.functional.binary_cross_entropy_with_logits(logits, labels.float())
        # Per-class targets
        targets = torch.where(labels == 1,
                              torch.tensor(self.gamma_1, device=logits.device),
                              torch.tensor(self.gamma_0, device=logits.device))
        penalty = self.lam * torch.log(1 + (logits.squeeze() - targets) ** 2).mean()
        return bce + penalty


class SigmaDampLoss(nn.Module):
    """σ-Damp loss with per-class temperatures.

    ℓ_σ-damp(y, f) = ℓ_log(T_y * 1.278 * yf * (1 - σ(1.278 * yf)))
    """
    def __init__(self, T_0=1.0, T_1=2.0):
        super().__init__()
        self.T_0 = T_0
        self.T_1 = T_1

    def forward(self, logits, labels):
        # Convert to margin: m = (2*y - 1) * f for y in {0,1}
        signs = 2 * labels.float() - 1  # {-1, +1}
        margins = signs * logits.squeeze()

        # Per-class temperatures
        T = torch.where(labels == 1,
                        torch.tensor(self.T_1, device=logits.device),
                        torch.tensor(self.T_0, device=logits.device))

        # Damped input: T_y * 1.278 * m * (1 - σ(1.278 * m))
        scaled_m = 1.278 * margins
        damped_input = T * scaled_m * (1 - torch.sigmoid(scaled_m))

        # Log-loss on damped input
        loss = torch.log(1 + torch.exp(-damped_input)).mean()
        return loss


class SpurBreastDataset(Dataset):
    """
    PyTorch Dataset that loads images on-the-fly with shortcut metadata.
    This is needed for finetuning — you can't backprop through frozen embeddings.
    """
    def __init__(self, image_paths, labels, z1_labels, z2_labels, groups, transform=None):
        self.image_paths = image_paths
        self.labels = labels            # y in {0, 1}
        self.z1_labels = z1_labels      # field strength shortcut
        self.z2_labels = z2_labels      # flip shortcut
        self.groups = groups            # 4 groups from (z1_agree, z2_agree)
        self.transform = transform or transforms.Compose([
            transforms.Resize((224, 224)),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406],
                                 std=[0.229, 0.224, 0.225])
        ])

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        img = Image.open(self.image_paths[idx]).convert('RGB')

        # Apply the vertical flip if z2=1 (this must happen BEFORE transforms)
        if self.z2_labels[idx] == 1:
            img = img.transpose(Image.FLIP_TOP_BOTTOM)

        img = self.transform(img)
        return img, self.labels[idx], self.groups[idx]


class SpurBreastTestDataset(Dataset):
    """Test dataset — no shortcuts applied, just clean images."""
    def __init__(self, image_paths, labels, transform=None):
        self.image_paths = image_paths
        self.labels = labels
        self.transform = transform or transforms.Compose([
            transforms.Resize((224, 224)),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406],
                                 std=[0.229, 0.224, 0.225])
        ])

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        img = Image.open(self.image_paths[idx]).convert('RGB')
        img = self.transform(img)
        # groups for test = just the label (benign vs malignant)
        return img, self.labels[idx], self.labels[idx]

def prepare_spurbreast_datasets(fs_dir, baseline_dir, clinical_xlsx_path,
                                 rho2=0.9, flip_seed=42):
    """
    Prepare SpurBreast datasets for finetuning.

    Instead of extracting embeddings, this returns Dataset objects
    that load images on-the-fly so gradients flow through the backbone.
    """
    fs_map = load_duke_field_strength(clinical_xlsx_path)

    def get_fs_label(filename):
        try:
            pat_num = int(os.path.basename(filename).split('-')[1])
            return fs_map.get(pat_num, 0)
        except:
            return 0

    # ── Collect training paths + metadata ──
    train_dir = os.path.join(fs_dir, 'training')
    pos_paths = get_file_paths(os.path.join(train_dir, '1'), 1)
    neg_paths = get_file_paths(os.path.join(train_dir, '0'), 0)
    all_paths = pos_paths + neg_paths

    rng = np.random.RandomState(flip_seed)
    image_paths, y_list, z1_list, z2_list = [], [], [], []

    for path, label in all_paths:
        fname = os.path.basename(path)
        z1_val = get_fs_label(fname)
        z2_val = label if rng.random() < rho2 else 1 - label

        image_paths.append(path)
        y_list.append(label)
        z1_list.append(z1_val)
        z2_list.append(z2_val)

    y_arr = np.array(y_list, dtype=np.int64)
    z1_arr = np.array(z1_list, dtype=np.int64)
    z2_arr = np.array(z2_list, dtype=np.int64)

    # Step 2: bias field strength correlation
    image_paths, y_arr, z1_arr, z2_arr = bias_field_strength(
        image_paths, y_arr, z1_arr, z2_arr, rho1=0.9, seed=flip_seed
    )

    # Step 3: recompute groups after biasing
    agree1 = (y_arr == z1_arr).astype(int)
    agree2 = (y_arr == z2_arr).astype(int)
    groups_arr = 2 * agree1 + agree2

    print(f"Training: {len(y_arr)} images")
    print(f"  z1 (field strength) correlation: {np.mean(agree1):.3f}")
    print(f"  z2 (vertical flip) correlation:  {np.mean(agree2):.3f}")
    print(f"  Groups: {dict(zip(*np.unique(groups_arr, return_counts=True)))}")

    # 80/20 train/val split
    np.random.seed(42)
    indices = np.random.permutation(len(y_arr))
    split = int(len(y_arr) * 0.8)
    tr_idx, val_idx = indices[:split], indices[split:]

    # NOTE: z2 flip is applied inside the Dataset.__getitem__, not during extraction
    train_dataset = SpurBreastDataset(
        [image_paths[i] for i in tr_idx],
        y_arr[tr_idx], z1_arr[tr_idx], z2_arr[tr_idx], groups_arr[tr_idx]
    )

    # ── Test set (baseline, unbiased) ──
    test_dir = os.path.join(baseline_dir, 'test')
    test_pos = get_file_paths(os.path.join(test_dir, '1'), 1)
    test_neg = get_file_paths(os.path.join(test_dir, '0'), 0)
    test_all = test_pos + test_neg
    test_paths = [p for p, _ in test_all]
    test_labels = np.array([l for _, l in test_all], dtype=np.int64)
    # In prepare_spurbreast_datasets, split the test set:
    test_dataset = SpurBreastTestDataset(test_paths, test_labels)

    # Use 20% of test as unbiased validation for early stopping
    np.random.seed(99)
    test_indices = np.random.permutation(len(test_labels))
    val_split = int(len(test_labels) * 0.2)

    unbiased_val_dataset = SpurBreastTestDataset(
        [test_paths[i] for i in test_indices[:val_split]],
        test_labels[test_indices[:val_split]]
    )
    final_test_dataset = SpurBreastTestDataset(
        [test_paths[i] for i in test_indices[val_split:]],
        test_labels[test_indices[val_split:]]
    )
    print(f"Test: {len(test_labels)} images")

    return train_dataset, unbiased_val_dataset, final_test_dataset


def bias_field_strength(image_paths, y_arr, z1_arr, z2_arr, rho1=0.9, seed=42):
    """
    Subsample training data so that field strength (z1) correlates
    with label (y) at rate rho1.

    Keeps samples where z1==y with probability rho1,
    and samples where z1!=y with probability (1-rho1),
    then balances to get the target correlation.
    """
    rng = np.random.RandomState(seed)

    # Split into z1-agrees-with-y and z1-disagrees-with-y
    agree_idx = np.where(y_arr == z1_arr)[0]
    disagree_idx = np.where(y_arr != z1_arr)[0]

    # Target: rho1 fraction should be agree, (1-rho1) should be disagree
    # Figure out how many of each we need
    # Use the smaller constraint to set total size
    n_agree_available = len(agree_idx)
    n_disagree_available = len(disagree_idx)

    # Option A: keep all agrees, subsample disagrees
    n_agree_from_disagree = int(n_agree_available * (1 - rho1) / rho1)
    # Option B: keep all disagrees, subsample agrees
    n_disagree_from_agree = int(n_disagree_available * rho1 / (1 - rho1))

    if n_agree_from_disagree <= n_disagree_available:
        # Keep all agrees, subsample disagrees
        keep_agree = agree_idx
        keep_disagree = rng.choice(disagree_idx, size=n_agree_from_disagree, replace=False)
    else:
        # Keep all disagrees, subsample agrees
        keep_agree = rng.choice(agree_idx, size=n_disagree_from_agree, replace=False)
        keep_disagree = disagree_idx

    keep_idx = np.concatenate([keep_agree, keep_disagree])
    rng.shuffle(keep_idx)

    new_paths = [image_paths[i] for i in keep_idx]
    new_y = y_arr[keep_idx]
    new_z1 = z1_arr[keep_idx]
    new_z2 = z2_arr[keep_idx]

    actual_corr = np.mean(new_y == new_z1)
    print(f"  Field strength biasing: {len(y_arr)} -> {len(new_y)} images")
    print(f"  z1 correlation: {np.mean(y_arr == z1_arr):.3f} -> {actual_corr:.3f}")

    return new_paths, new_y, new_z1, new_z2

#Finetuning functions

def create_resnet_model(device='cuda'):
    """Fresh pretrained ResNet-50 with a single output head."""
    model = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)
    model.fc = nn.Linear(2048, 1)
    return model.to(device)

from torch.utils.data import Dataset, DataLoader, Subset

def get_subset_by_indices(dataset, indices):
    """Create a Subset from a SpurBreastDataset using given indices."""
    return Subset(dataset, indices)


def subsample_dataset(dataset, strategy='full', seed=0):
    """
    Apply subsampling strategy to a SpurBreastDataset.
    Returns a Subset with the appropriate indices.
    """
    groups = dataset.groups
    n = len(groups)
    rng = np.random.RandomState(seed)

    if strategy == 'full':
        return dataset

    elif strategy == 'gb_down':
        unique_groups = np.unique(groups)
        min_size = min(np.sum(groups == g) for g in unique_groups)
        indices = []
        for g in unique_groups:
            g_idx = np.where(groups == g)[0]
            chosen = rng.choice(g_idx, size=min_size, replace=False)
            indices.extend(chosen)
        rng.shuffle(indices)
        return get_subset_by_indices(dataset, indices)

    elif strategy.startswith('gb_down_'):
        # GB-Down after first randomly keeping a fraction of data
        frac = int(strategy.split('_')[2]) / 100
        # First randomly subsample
        n_keep = max(int(n * frac), 2)
        pre_idx = rng.choice(n, size=n_keep, replace=False)
        pre_groups = groups[pre_idx]
        # Then balance within the subsample
        unique_groups = np.unique(pre_groups)
        min_size = min(np.sum(pre_groups == g) for g in unique_groups)
        if min_size == 0:
            return get_subset_by_indices(dataset, pre_idx)
        indices = []
        for g in unique_groups:
            g_idx = np.where(pre_groups == g)[0]
            chosen = rng.choice(g_idx, size=min_size, replace=False)
            indices.extend(chosen)
        rng.shuffle(indices)
        final_idx = pre_idx[indices]
        return get_subset_by_indices(dataset, final_idx)

    elif strategy == 'gb_over':
        unique_groups = np.unique(groups)
        max_size = max(np.sum(groups == g) for g in unique_groups)
        indices = []
        for g in unique_groups:
            g_idx = np.where(groups == g)[0]
            chosen = rng.choice(g_idx, size=max_size, replace=True)
            indices.extend(chosen)
        rng.shuffle(indices)
        return get_subset_by_indices(dataset, indices)

    elif strategy.startswith('random_'):
        ratio = int(strategy.split('_')[1]) / 100
        n_keep = max(int(n * ratio), 2)
        indices = rng.choice(n, size=n_keep, replace=False)
        return get_subset_by_indices(dataset, indices)

    elif strategy == 'importance_weighted':
        unique_groups = np.unique(groups)
        # Target: equal number from each group, total size = n
        n_per_group = n // len(unique_groups)
        indices = []
        for g in unique_groups:
            g_idx = np.where(groups == g)[0]
            chosen = rng.choice(g_idx, size=n_per_group, replace=True)
            indices.extend(chosen)
        rng.shuffle(indices)
        return get_subset_by_indices(dataset, indices)
    # elif strategy == 'importance_weighted':
    #     # Upweight minority groups by oversampling proportional to 1/group_size
    #     unique_groups = np.unique(groups)
    #     group_sizes = {g: np.sum(groups == g) for g in unique_groups}
    #     total = sum(1.0 / s for s in group_sizes.values())

    #     # Target total size = original n, but rebalanced
    #     target_n = n
    #     indices = []
    #     for g in unique_groups:
    #         g_idx = np.where(groups == g)[0]
    #         # Each group gets samples proportional to 1/group_size
    #         n_samples = int(target_n * (1.0 / group_sizes[g]) / total)
    #         chosen = rng.choice(g_idx, size=n_samples, replace=True)
    #         indices.extend(chosen)
    #     rng.shuffle(indices)
    #     return get_subset_by_indices(dataset, indices)

    else:
        raise ValueError(f"Unknown strategy: {strategy}")

def get_shortcut_ratio(dataset):
    """Compute fraction of samples where shortcut agrees with label."""
    if isinstance(dataset, Subset):
        parent = dataset.dataset
        indices = dataset.indices
        groups = parent.groups[indices]
    else:
        groups = dataset.groups

    # Groups 2 and 3 have z1 agreeing with y (agree1=1)
    # Groups 1 and 3 have z2 agreeing with y (agree2=1)
    z1_agree = np.mean(np.isin(groups, [2, 3]))
    z2_agree = np.mean(np.isin(groups, [1, 3]))
    return z1_agree, z2_agree

def evaluate_model(model, loader, device='cuda'):
    """Evaluate model, return avg acc, WGA, and per-group accs."""
    model.eval()
    all_preds, all_labels, all_groups = [], [], []

    with torch.no_grad():
        for x, y, g in loader:
            x = x.to(device)
            logits = model(x).squeeze(-1)
            preds = (logits > 0).long().cpu()
            all_preds.append(preds)
            all_labels.append(y)
            all_groups.append(g)

    preds = torch.cat(all_preds)
    labels = torch.cat(all_labels).long()
    groups = torch.cat(all_groups)

    correct = (preds == labels).float()
    avg_acc = correct.mean().item()

    group_accs = {}
    for g in groups.unique():
        mask = groups == g
        if mask.sum() > 0:
            group_accs[g.item()] = correct[mask].mean().item()

    wga = min(group_accs.values()) if group_accs else 0.0
    return avg_acc, wga, group_accs


def train_finetune_spurbreast(model, train_loader, val_loader, loss_fn,
                               lr=1e-5, weight_decay=0.1, epochs=25,
                               patience=10, device='cuda'):
    """
    Full finetuning with early stopping on validation WGA.
    Uses lower lr than linear probe since we're updating the whole backbone.
    """
    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    best_val_wga = -1
    best_state = None
    patience_counter = 0

    for epoch in range(epochs):
        # ── Train ──
        model.train()
        total_loss = 0
        n_samples = 0
        for x, y, g in train_loader:
            x, y = x.to(device), y.to(device).float()
            optimizer.zero_grad()
            logits = model(x).squeeze(-1)
            loss = loss_fn(logits, y)
            loss.backward()
            optimizer.step()
            total_loss += loss.item() * x.size(0)
            n_samples += x.size(0)

        # ── Validate ──
        val_avg, val_wga, _ = evaluate_model(model, val_loader, device)

        if (epoch + 1) % 5 == 0:
            print(f"  Epoch {epoch+1}/{epochs} | "
                  f"train_loss={total_loss/n_samples:.4f} | "
                  f"val_avg={val_avg:.3f} | val_wga={val_wga:.3f}")

        if val_wga > best_val_wga:
            best_val_wga = val_wga
            best_state = copy.deepcopy(model.state_dict())
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= patience:
                print(f"  Early stopping at epoch {epoch+1}")
                break

    if best_state is not None:
        model.load_state_dict(best_state)

    return model, best_val_wga

def run_spurbreast_finetuning(train_dataset, val_dataset, test_dataset,
                               device='cuda'):
    val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=2)
    test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, num_workers=2)

    d = 2048  # ResNet-50 output dimension

    methods = {
        'ERM':      LogLoss(),
        'SD':       SpectralDecoupling(lam=0.05),
        'Marg-Log': MargLogLoss(lam=0.5, gamma_0=0, gamma_1=2),
        'σ-damp':   SigmaDampLoss(T_0=1, T_1=0.5),
    }

    conditions = [
        #Random: 5 points (ρ stays ~0.9, only n changes)
        'random_005',
        'random_010',
        'random_025',
        'random_050',
        'full',
        #Group-balanced: 5 points (ρ forced to 0.5, different n)
        'gb_down',            # balance full data (smallest n)
        'gb_down_025',        # balance 25% of data
        'gb_down_050',        # balance 50% of data
        'gb_down_075',        # balance 75% of data
        'gb_over',            # oversample (largest n)
        # Extra
        'importance_weighted',
    ]

    all_results = []

    for cond in conditions:
        for method_name, loss_fn in methods.items():
            for seed in range(2):
                print(f"\n{'='*60}")
                print(f"  {cond} | {method_name} | seed={seed}")
                print(f"{'='*60}")

                sub_dataset = subsample_dataset(train_dataset, strategy=cond, seed=seed)
                n_train = len(sub_dataset)
                d_over_n = d / n_train
                z1_rho, z2_rho = get_shortcut_ratio(sub_dataset)

                print(f"  n_train={n_train}, d/n={d_over_n:.3f}")
                print(f"  z1 shortcut-label ratio={z1_rho:.3f}, z2={z2_rho:.3f}")

                train_loader = DataLoader(sub_dataset, batch_size=64,
                                          shuffle=True, num_workers=4)

                torch.manual_seed(seed)
                model = create_resnet_model(device)

                model, val_wga = train_finetune_spurbreast(
                    model, train_loader, val_loader, loss_fn,
                    lr=1e-5,
                    weight_decay=0.1,
                    epochs=50,
                    patience=20,
                    device=device
                )

                test_avg, test_wga, test_groups = evaluate_model(
                    model, test_loader, device
                )

                print(f"  => test_avg={test_avg:.3f} | test_wga={test_wga:.3f}")

                all_results.append({
                    'condition': cond,
                    'method': method_name,
                    'seed': seed,
                    'n_train': n_train,
                    'd': d,
                    'd_over_n': d_over_n,
                    'z1_shortcut_ratio': z1_rho,
                    'z2_shortcut_ratio': z2_rho,
                    'val_wga': val_wga,
                    'test_wga': test_wga,
                    'test_avg': test_avg,
                })

    return pd.DataFrame(all_results)

def run_pca_experiment(X_train, y_train, groups_train,
                        X_test, y_test, groups_test,
                        group_names, dataset_name, loss_fns,
                        d_values=[32, 64, 128, 256, 512],
                        seeds=range(3)):
    """
    Run experiments at different PCA dimensions.
    Lower d concentrates variance into fewer dimensions,
    potentially creating concentrated shortcuts that MARG-CTRL can fix.
    """
    all_results = []

    # Standardise first (important for PCA)
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    for d in d_values:
        print(f"\n{'='*60}")
        print(f"  PCA d={d}")
        print(f"{'='*60}")

        pca = PCA(n_components=d)
        X_tr_pca = pca.fit_transform(X_train_scaled).astype(np.float32)
        X_te_pca = pca.transform(X_test_scaled).astype(np.float32)

        # How much variance is captured?
        var_explained = pca.explained_variance_ratio_.cumsum()[-1]
        print(f"  Variance explained: {var_explained:.3f}")
        print(f"  d/n ratio (full): {d/len(y_train):.4f}")

        df = run_real_data_experiment(
            X_tr_pca, y_train, groups_train,
            X_te_pca, y_test, groups_test,
            group_names, f'{dataset_name}_PCA{d}',
            loss_fns, seeds=seeds
        )
        df['pca_d'] = d
        df['var_explained'] = var_explained
        all_results.append(df)

    return pd.concat(all_results, ignore_index=True)


# run actual experiment
def run_real_data_experiment(X_train, y_train, groups_train,
                             X_test, y_test, groups_test,
                             group_names, dataset_name,
                             loss_fns, seeds=range(3)):
    d = X_train.shape[1]
    all_results = []

    for seed in seeds:
        Xgd, ygd, ggd, _ = real_data_gb_downsample(X_train, y_train, groups_train, seed=seed)
        Xgo, ygo, ggo, _ = real_data_gb_oversample(X_train, y_train, groups_train, seed=seed)
        Xrs, yrs, grs, _ = real_data_random_subsample(X_train, y_train, groups_train, ratio=0.25, seed=seed)# can try with diff ratios later too

        conditions = {
            'Full':            (X_train, y_train, groups_train),
            'GB-Down':         (Xgd, ygd, ggd),
            'GB-Over':         (Xgo, ygo, ggo),
            'Random (r=0.25)': (Xrs, yrs, grs),
        }

        for cond_name, (Xc, yc, gc) in conditions.items():
            for method_name, loss_factory in loss_fns.items():
                torch.manual_seed(seed)
                model = nn.Linear(d, 1, bias=False).to(device)
                optimizer = optim.Adam(model.parameters(), lr=0.01)

                Xt = torch.tensor(Xc, dtype=torch.float32).to(device)
                yt = torch.tensor(yc, dtype=torch.float32).to(device)

                loss_fn = loss_factory()
                for epoch in range(3000):
                    optimizer.zero_grad()
                    logits = model(Xt).squeeze()
                    loss = loss_fn(logits, yt)
                    loss.backward()
                    optimizer.step()

                save_checkpoint(model, {
                    'dataset': dataset_name, 'condition': cond_name,
                    'method': method_name, 'seed': seed,
                    'n_train': len(yc), 'd': d, 'd_over_n': d / len(yc),
                }, f'{dataset_name}_{cond_name}_{method_name}_seed{seed}')

                model.eval()
                with torch.no_grad():
                    train_logits = model(Xt).squeeze()
                    train_loss = loss_fn(train_logits, yt).item()
                    train_preds = np.sign(train_logits.cpu().numpy())
                    train_preds[train_preds == 0] = 1
                    train_acc = np.mean(train_preds == yc)
                print(f'  {dataset_name} | {cond_name} | {method_name}: 'f'train_loss={train_loss:.4f}, train_acc={train_acc:.4f}')

                with torch.no_grad():
                    Xte = torch.tensor(X_test, dtype=torch.float32).to(device)
                    test_logits = model(Xte).squeeze().cpu().numpy()
                    test_preds = np.sign(test_logits)
                    test_preds[test_preds == 0] = 1

                metrics = compute_group_metrics(test_preds, y_test, groups_test, group_names)
                print_group_summary(metrics, f'{dataset_name} | {cond_name} | {method_name}')

                result = {
                    'dataset': dataset_name, 'condition': cond_name,
                    'method': method_name, 'seed': seed,
                    'n_train': len(yc), 'd': d, 'd_over_n': d / len(yc),
                    'avg_acc': metrics['avg_acc'], 'wga': metrics['wga'],
                }
                for name, info in metrics['per_group'].items():
                    result[f'acc_{name}'] = info['accuracy']
                    result[f'n_{name}'] = info['size']

                all_results.append(result)

    return pd.DataFrame(all_results)

DUKE metadata: 922 patients, 1.5T=468, 3T=454


In [ ]:

WATERBIRDS = False
SPURBREAST = True

if WATERBIRDS:
    !pip install wilds
    splits, group_names = load_waterbirds()
    df_pca = run_pca_experiment(
        splits['train']['X'], splits['train']['y'], splits['train']['groups'],
        splits['test']['X'], splits['test']['y'], splits['test']['groups'],
        group_names, 'Waterbirds', LOSS_FNS_CORE,
        d_values=[32, 64, 128, 256, 512],
        seeds=range(1)
    )
    save_csv(df_pca, 'results_waterbirds_pca.csv')

    # df_wb = run_real_data_experiment(
    #     splits['train']['X'], splits['train']['y'], splits['train']['groups'],
    #     splits['test']['X'], splits['test']['y'], splits['test']['groups'],
    #     group_names, 'Waterbirds', LOSS_FNS_CORE)
    # save_csv(df_wb, 'results_waterbirds.csv')

if SPURBREAST: # label 1 = malignant, label 0 = benign
    # Copy data to local SSD first (run once)
    DRIVE_ROOT = '/content/drive/MyDrive/shortcut_experiments'
    os.makedirs(f'{DRIVE_ROOT}/csv', exist_ok=True)

    import shutil, os
    if not os.path.exists('/content/local_data/field_strength'):
        print("Copying data to local disk...")
        shutil.copytree('/content/drive/MyDrive/field_strength/field_strength', '/content/local_data/field_strength')
        shutil.copytree('/content/drive/MyDrive/baseline_low', '/content/local_data/baseline_low')
        shutil.copy('/content/drive/MyDrive/Clinical_and_Other_Features.xlsx', '/content/local_data/Clinical_and_Other_Features.xlsx')
        print("Done copying.")

    train_dataset, val_dataset, test_dataset = prepare_spurbreast_datasets(
        fs_dir='/content/local_data/field_strength',
        baseline_dir='/content/local_data/baseline_low',
        clinical_xlsx_path='/content/local_data/Clinical_and_Other_Features.xlsx',
        rho2=0.9,
        flip_seed=42,
    )
    # train_dataset, val_dataset, test_dataset = prepare_spurbreast_datasets(
    # fs_dir='/content/drive/MyDrive/field_strength/field_strength',
    # baseline_dir='/content/drive/MyDrive/baseline_low',
    # clinical_xlsx_path='/content/drive/MyDrive/Clinical_and_Other_Features.xlsx',
    # rho2=0.9,
    # flip_seed=42,
    # )

    # Run full finetuning experiments
    df_spurbreast_ft = run_spurbreast_finetuning(
        train_dataset, val_dataset, test_dataset,
        device='cuda'
    )

    # Save results
    df_spurbreast_ft.to_csv(f'{DRIVE_ROOT}/csv/results_spurbreast_finetuning.csv', index=False)

    # Print summary
    agg = df_spurbreast_ft.groupby(['condition', 'method']).agg(
        test_wga=('test_wga', 'mean'),
        test_wga_std=('test_wga', 'std'),
    ).reset_index()
    print(agg.to_string(index=False, float_format='{:.3f}'.format))

    # sb_splits, sb_group_names_train, sb_group_names_test = extract_spurbreast_features(
    #     fs_dir='/content/drive/MyDrive/field_strength/field_strength',
    #     vf_dir='/content/drive/MyDrive/vertical_flip',
    #     baseline_dir='/content/drive/MyDrive/baseline_low',
    #     rho2=0.9,       # flip correlation strength
    #     flip_seed=42,    # reproducibility
    # )

    # X_tr = sb_splits['train']['X']
    # y_tr = sb_splits['train']['y']
    # g_tr = sb_splits['train']['groups']

    # X_te = sb_splits['test']['X']
    # y_te = sb_splits['test']['y']
    # g_te = sb_splits['test']['groups']

    # print(f'Device: {device}')

    # df_spurbreast = run_real_data_experiment(
    #     X_tr, y_tr, g_tr,
    #     X_te, y_te, g_te,
    #     group_names=sb_group_names_test,
    #     dataset_name='SpurBreast',
    #     loss_fns=LOSS_FNS_CORE,
    #     seeds=range(3),
    # )

    # save_csv(df_spurbreast, 'results_spurbreast.csv')

DUKE metadata: 922 patients, 1.5T=468, 3T=454
  Field strength biasing: 9563 -> 5448 images
  z1 correlation: 0.513 -> 0.900
Training: 5448 images
  z1 (field strength) correlation: 0.900
  z2 (vertical flip) correlation:  0.902
  Groups: {np.int64(0): np.int64(50), np.int64(1): np.int64(494), np.int64(2): np.int64(483), np.int64(3): np.int64(4421)}
Test: 6788 images

  gb_over | ERM | seed=0
  n_train=14192, d/n=0.144
  z1 shortcut-label ratio=0.500, z2=0.500
Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /root/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 217MB/s]


  Epoch 5/50 | train_loss=0.0712 | val_avg=0.561 | val_wga=0.389
  Epoch 10/50 | train_loss=0.0163 | val_avg=0.564 | val_wga=0.411
  Epoch 15/50 | train_loss=0.0104 | val_avg=0.555 | val_wga=0.366
  Epoch 20/50 | train_loss=0.0076 | val_avg=0.560 | val_wga=0.388
  Early stopping at epoch 21
  => test_avg=0.627 | test_wga=0.596

  gb_over | ERM | seed=1
  n_train=14192, d/n=0.144
  z1 shortcut-label ratio=0.500, z2=0.500
  Epoch 5/50 | train_loss=0.0771 | val_avg=0.549 | val_wga=0.319
  Epoch 10/50 | train_loss=0.0173 | val_avg=0.556 | val_wga=0.336
  Epoch 15/50 | train_loss=0.0105 | val_avg=0.547 | val_wga=0.374
  Epoch 20/50 | train_loss=0.0061 | val_avg=0.563 | val_wga=0.352
  Early stopping at epoch 21
  => test_avg=0.570 | test_wga=0.468

  gb_over | SD | seed=0
  n_train=14192, d/n=0.144
  z1 shortcut-label ratio=0.500, z2=0.500
  Epoch 5/50 | train_loss=0.3396 | val_avg=0.558 | val_wga=0.379
  Epoch 10/50 | train_loss=0.3218 | val_avg=0.566 | val_wga=0.394
  Epoch 15/50 | train_